In [5]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import shap
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

# ==========================================
# 1. 数据集定义 (缺失值自动填补 + 断层检测)
# ==========================================
class TimeAwareMultiStationDataset(Dataset):
    def __init__(self, filepaths, seq_len=24, target_col='GPP_DT_VUT_REF', time_col='date',
                 forcing_cols=None, state_cols=None,
                 static_cols=['Lat', 'Long'], lc_col='Veg_ID',
                 feat_min_f=None, feat_max_f=None, feat_min_s=None, feat_max_s=None,
                 static_min=None, static_max=None, split_type='train'):

        self.seq_len = seq_len
        self.samples = []

        self.station_forcing = []
        self.station_state = []
        self.station_time_features = []
        self.station_targets = []
        self.station_dates = []

        self.station_static = []
        self.station_lc = []

        for filepath in filepaths:
            data = pd.read_csv(filepath)
            if time_col not in data.columns:
                print(f"⚠️ 在文件 {filepath} 中找不到时间列 '{time_col}'，跳过。")
                continue

            # 异常值清洗与插值填补 NaN
            data = data.replace([-9999, -9999.0, -999], np.nan)
            cols_to_clean = forcing_cols + state_cols + [target_col]
            cols_exist = [c for c in cols_to_clean if c in data.columns]

            data[cols_exist] = data[cols_exist].interpolate(method='linear', limit_direction='both')
            data = data.dropna(subset=cols_exist).reset_index(drop=True)

            data[time_col] = pd.to_datetime(data[time_col])
            data = data.sort_values(by=time_col).reset_index(drop=True)

            if len(data) < seq_len:
                continue

            dates = data[time_col]
            self.station_dates.append(dates.values)

            hours = dates.dt.hour.values
            days = dates.dt.dayofyear.values
            time_feats = np.column_stack([
                np.sin(2 * np.pi * hours / 24.0), np.cos(2 * np.pi * hours / 24.0),
                np.sin(2 * np.pi * days / 365.25), np.cos(2 * np.pi * days / 365.25)
            ])

            forcing_data = data[forcing_cols].values
            state_data = data[state_cols].values
            static_data = data[static_cols].values
            lc_data = data[lc_col].values.astype(np.int64)

            if target_col in data.columns:
                targets = data[target_col].values
            else:
                targets = data.iloc[:, -1].values

            self.station_forcing.append(forcing_data)
            self.station_state.append(state_data)
            self.station_time_features.append(time_feats)
            self.station_targets.append(targets)
            self.station_static.append(static_data)
            self.station_lc.append(lc_data)

        if not self.station_forcing:
            raise ValueError(f"加载 {split_type} 数据失败，可能所有数据均被清洗掉或长度不足。")

        all_forcing_concat = np.vstack(self.station_forcing)
        all_state_concat = np.vstack(self.station_state)
        all_static_concat = np.vstack(self.station_static)

        self.feat_min_f = np.min(all_forcing_concat, axis=0) if feat_min_f is None else feat_min_f
        self.feat_max_f = np.max(all_forcing_concat, axis=0) if feat_max_f is None else feat_max_f
        self.feat_min_s = np.min(all_state_concat, axis=0) if feat_min_s is None else feat_min_s
        self.feat_max_s = np.max(all_state_concat, axis=0) if feat_max_s is None else feat_max_s

        self.static_min = np.min(all_static_concat, axis=0) if static_min is None else static_min
        self.static_max = np.max(all_static_concat, axis=0) if static_max is None else static_max

        # 归一化与样本切分 (包含时间连续性断层检测)
        for i in range(len(self.station_forcing)):
            self.station_forcing[i] = (self.station_forcing[i] - self.feat_min_f) / (self.feat_max_f - self.feat_min_f + 1e-8)
            self.station_state[i] = (self.station_state[i] - self.feat_min_s) / (self.feat_max_s - self.feat_min_s + 1e-8)
            self.station_static[i] = (self.station_static[i] - self.static_min) / (self.static_max - self.static_min + 1e-8)

            dates_array = self.station_dates[i]
            if len(dates_array) > 1:
                diffs = pd.Series(dates_array).diff()
                mode_step = diffs.mode()[0]
                expected_duration = mode_step * (self.seq_len - 1)

                num_samples = len(self.station_forcing[i]) - self.seq_len + 1
                if num_samples > 0:
                    for j in range(num_samples):
                        actual_duration = pd.Timedelta(dates_array[j + self.seq_len - 1] - dates_array[j])
                        if actual_duration == expected_duration:
                            self.samples.append((i, j))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        station_idx, start_idx = self.samples[idx]

        x_forcing = self.station_forcing[station_idx][start_idx : start_idx + self.seq_len]
        x_state = self.station_state[station_idx][start_idx : start_idx + self.seq_len]
        time_x = self.station_time_features[station_idx][start_idx : start_idx + self.seq_len]
        y = self.station_targets[station_idx][start_idx + self.seq_len - 1]
        target_date = self.station_dates[station_idx][start_idx + self.seq_len - 1]

        x_static = self.station_static[station_idx][start_idx : start_idx + self.seq_len]
        x_lc = self.station_lc[station_idx][start_idx : start_idx + self.seq_len]

        return (
            torch.tensor(x_forcing, dtype=torch.float32),
            torch.tensor(x_state, dtype=torch.float32),
            torch.tensor(time_x, dtype=torch.float32),
            torch.tensor(x_static, dtype=torch.float32),
            torch.tensor(x_lc, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            str(target_date)
        )

# ==========================================
# 2. TCN 基础模块定义
# ==========================================
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1, self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

# ==========================================
# 3. 交叉注意力融合模型
# ==========================================
class TCN_Transformer_CrossAttention(nn.Module):
    def __init__(self, num_forcing_features, num_state_features, seq_len,
                 num_static=2, num_lc_classes=10, lc_embed_dim=8,
                 d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super(TCN_Transformer_CrossAttention, self).__init__()

        self.tcn = TemporalConvNet(num_inputs=num_forcing_features,
                                   num_channels=[d_model] * 6,
                                   kernel_size=3, dropout=dropout)

        self.lc_embedding = nn.Embedding(num_embeddings=num_lc_classes, embedding_dim=lc_embed_dim)

        combined_state_dim = num_state_features + num_static + lc_embed_dim
        self.state_linear = nn.Linear(combined_state_dim, d_model)
        self.time_projector = nn.Linear(4, d_model)

        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.cross_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4), nn.ReLU(),
            nn.Linear(d_model // 4, 1)
        )

    def forward(self, x_forcing, x_state, time_x, x_static, x_lc):
        x_tcn_in = x_forcing.transpose(1, 2)
        f_tcn = self.tcn(x_tcn_in)
        f_met_memory = f_tcn.transpose(1, 2)

        lc_emb = self.lc_embedding(x_lc)
        combined_state = torch.cat([x_state, x_static, lc_emb], dim=-1)

        x_s_emb = self.state_linear(combined_state)
        time_emb = self.time_projector(time_x)
        x_state_combined = x_s_emb + time_emb
        f_state_global = self.transformer_encoder(x_state_combined)

        fused_features, _ = self.cross_attention(
            query=f_state_global,
            key=f_met_memory,
            value=f_met_memory
        )

        last_step_features = fused_features[:, -1, :]
        out = self.regressor(last_step_features)
        return out.squeeze(-1)

# ==========================================
# 4. SHAP 分析代码
# ==========================================
def perform_shap_analysis(model, dataloader, device, output_img_folder, forcing_cols, state_cols):
    print("\n🔍 开始执行 SHAP 变量重要性分析...")
    model.eval()

    shap_loader = DataLoader(dataloader.dataset, batch_size=4000, shuffle=True)
    batch = next(iter(shap_loader))

    batch_forcing, batch_state, batch_time = batch[0].to(device), batch[1].to(device), batch[2].to(device)
    batch_static, batch_lc = batch[3].to(device), batch[4].to(device)

    bg_size, test_size = 1000, 1000
    if batch_forcing.size(0) < (bg_size + test_size):
        print("⚠️ Batch size 太小，跳过 SHAP 分析。")
        return

    bg_f, bg_s = batch_forcing[:bg_size], batch_state[:bg_size]
    test_f, test_s = batch_forcing[bg_size:bg_size+test_size], batch_state[bg_size:bg_size+test_size]

    class SHAPWrapper(nn.Module):
        def __init__(self, base_model, t_ref, st_ref, lc_ref):
            super().__init__()
            self.base_model = base_model
            self.t_ref = t_ref[:1]
            self.st_ref = st_ref[:1]
            self.lc_ref = lc_ref[:1]

        def forward(self, x_f, x_s):
            b_size = x_f.size(0)
            t_x = self.t_ref.expand(b_size, -1, -1)
            x_st = self.st_ref.expand(b_size, -1, -1)
            x_l = self.lc_ref.expand(b_size, -1)
            out = self.base_model(x_f, x_s, t_x, x_st, x_l)
            return out.unsqueeze(-1)

    wrapper_model = SHAPWrapper(model, batch_time, batch_static, batch_lc).to(device)
    wrapper_model.eval()

    explainer = shap.GradientExplainer(wrapper_model, [bg_f, bg_s])
    shap_values = explainer.shap_values([test_f, test_s])

    if isinstance(shap_values, list) and len(shap_values) == 1 and isinstance(shap_values[0], list):
        shap_values = shap_values[0]

    shap_forcing = np.array(shap_values[0])
    shap_state = np.array(shap_values[1])

    shap_forcing_2d = shap_forcing.reshape(-1, len(forcing_cols))
    shap_state_2d = shap_state.reshape(-1, len(state_cols))

    test_f_2d = test_f.cpu().numpy().reshape(-1, len(forcing_cols))
    test_s_2d = test_s.cpu().numpy().reshape(-1, len(state_cols))

    shap_combined = np.concatenate([shap_forcing_2d, shap_state_2d], axis=1)
    features_combined = np.concatenate([test_f_2d, test_s_2d], axis=1)
    feature_names = forcing_cols + state_cols

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
    plt.title("SHAP Summary: Global Feature Impact on GPP (Time-Flattened)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Summary_Plot.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)
    plt.title("SHAP Global Feature Importance (Magnitude)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Bar_Plot.png"), dpi=300)
    plt.close()

    print(f"✅ SHAP 生成完成，保存至: {output_img_folder}")

# ==========================================
# 5. 主流程
# ==========================================
def train_time_aware_model():
    seq_len = 96
    batch_size = 64
    num_epochs = 100
    learning_rate = 0.001
    patience = 10
    TIME_COLUMN_NAME = 'date'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data_folder = r"C:\Users\Admin\Desktop\实验三\全波段全变量DT"
    output_img_folder = os.path.join(data_folder, "Results_TCN_Transformerkua分地类")
    os.makedirs(output_img_folder, exist_ok=True)

    forcing_cols = ['SW_IN_F', 'SW_IN_POT', 'CO2_F_MDS', 'P_F', 'VPD_F', 'TA_F', 'TS_F_MDS_1', 'SWC_F_MDS_1', 'WS_F']
    state_cols = ['EPIC_Available_Mask', 'Band317nm_Ref', 'Band325nm_Ref', 'Band340nm_Ref',
                  'Band388nm_Ref', 'Band443nm_Ref', 'Band551nm_Ref', 'Band680nm_Ref',
                  'Band688nm_Ref', 'Band764nm_Ref', 'Band780nm_Ref', 'Mean_SZA', 'Mean_VZA', 'Mean_RAA']
    static_cols = ['Lat', 'Long']
    lc_col = 'Veg_ID'

    test_sites = ['US-xST_DBF', 'US-Bar_DBF', 'US-HB3_ENF', 'US-xNW_ENF', 'US-xKZ_GRA', 'US-xCP_GRA', 'US-A39_CRO', 'US-xJR_OSH', 'US-xTL_WET', 'US-SRM_WSA']
    val_sites = ['US-UMB_DBF', 'US-xBR_DBF', 'CA-TP1_ENF', 'US-xRM_ENF', 'US-Kon_GRA', 'US-xAE_GRA', 'US-Ne1_CRO', 'CA-Mer_WET', 'US-Whs_OSH', 'US-xDS_CVM']

    all_files = glob.glob(os.path.join(data_folder, "*.[cC][sS][vV]"))
    if not all_files:
        print("❌ 错误：未在指定目录找到CSV文件。")
        return

    train_files, val_files, test_files = [], [], []
    for f in all_files:
        fname = os.path.basename(f)
        if any(site in fname for site in test_sites):
            test_files.append(f)
        elif any(site in fname for site in val_sites):
            val_files.append(f)
        else:
            train_files.append(f)

    print(f"📁 共找到 {len(all_files)} 个站点文件。")
    print(f"   -> 训练集: {len(train_files)} | 验证集: {len(val_files)} | 测试集: {len(test_files)}")

    train_dataset = TimeAwareMultiStationDataset(
        train_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col, split_type='train'
    )

    f_min_f, f_max_f = train_dataset.feat_min_f, train_dataset.feat_max_f
    f_min_s, f_max_s = train_dataset.feat_min_s, train_dataset.feat_max_s
    s_min, s_max = train_dataset.static_min, train_dataset.static_max

    val_dataset = TimeAwareMultiStationDataset(
        val_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col,
        feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
        static_min=s_min, static_max=s_max, split_type='val'
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = TCN_Transformer_CrossAttention(
        num_forcing_features=len(forcing_cols),
        num_state_features=len(state_cols),
        seq_len=seq_len,
        num_static=len(static_cols),
        num_lc_classes=10,
        lc_embed_dim=8
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    checkpoint_latest_path = os.path.join(output_img_folder, "checkpoint_latest.pth")
    checkpoint_best_path = os.path.join(output_img_folder, "checkpoint_best.pth")

    start_epoch = 0
    best_val_loss = float('inf')
    epochs_no_improve = 0

    if os.path.exists(checkpoint_latest_path):
        print(f"\n🔄 检测到本地进度，尝试恢复...")
        try:
            checkpoint = torch.load(checkpoint_latest_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_loss = checkpoint['best_val_loss']
            epochs_no_improve = checkpoint['epochs_no_improve']
            print(f"✅ 从第 {start_epoch} 轮恢复，历史最佳 MSE: {best_val_loss:.4f}")
        except Exception as e:
            print(f"⚠️ 无法加载历史进度，将重新开始。错误信息: {e}")

    if start_epoch < num_epochs and epochs_no_improve < patience:
        print(f"🚀 开始训练...\n" + "-"*40)
        for epoch in range(start_epoch, num_epochs):
            model.train()
            train_loss = 0

            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in train_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                optimizer.zero_grad()
                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                loss = criterion(outputs, batch_y)
                loss.backward()

                # 梯度裁剪
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in val_loader:
                    batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                    batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                    outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            print(f"Epoch [{epoch+1:03d}/{num_epochs}] | Train MSE: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f}")

            if avg_val_loss < best_val_loss:
                print(f"   🌟 新的最佳模型！MSE: {avg_val_loss:.4f}。")
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), checkpoint_best_path)
            else:
                epochs_no_improve += 1
                print(f"   ⏳ 验证集未改善 (早停: {epochs_no_improve}/{patience})")

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'epochs_no_improve': epochs_no_improve
            }, checkpoint_latest_path)

            if epochs_no_improve >= patience:
                print(f"\n🛑 早停机制触发，训练结束。")
                break
            print("-" * 40)

# ==========================================
    # 6. 分站点独立测试评估 (整合高级绘图逻辑)
    # ==========================================
    if os.path.exists(checkpoint_best_path):
        print(f"\n🎯 开始测试集评估...")
        model.load_state_dict(torch.load(checkpoint_best_path, map_location=device))
    model.eval()

    global_all_preds, global_all_targets = [], []
    for test_file in test_files:
        station_name = os.path.basename(test_file).replace('.csv', '')
        single_test_dataset = TimeAwareMultiStationDataset(
            [test_file], seq_len, time_col=TIME_COLUMN_NAME,
            forcing_cols=forcing_cols, state_cols=state_cols,
            static_cols=static_cols, lc_col=lc_col,
            feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
            static_min=s_min, static_max=s_max, split_type='test'
        )

        if len(single_test_dataset) == 0: continue

        single_test_loader = DataLoader(single_test_dataset, batch_size=batch_size, shuffle=False)
        all_preds, all_targets, all_times = [], [], []

        with torch.no_grad():
            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, batch_dt in single_test_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc = batch_static.to(device), batch_lc.to(device)

                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(batch_y.numpy())
                all_times.extend(batch_dt)

        all_preds, all_targets = np.array(all_preds), np.array(all_targets)
        all_times = pd.to_datetime(all_times)

        # 应用数据清洗要求：剔除 GPP < 0 的异常值
        valid_mask = all_targets >= 0
        plot_targets = all_targets[valid_mask]
        plot_preds = all_preds[valid_mask]
        plot_times = all_times[valid_mask]

        if len(plot_targets) == 0:
            continue

        global_all_preds.extend(plot_preds)
        global_all_targets.extend(plot_targets)

        station_mse = np.mean((plot_preds - plot_targets)**2)
        station_r2 = r2_score(plot_targets, plot_preds)

        print(f"📢 站点: {station_name} | 测试集 MSE: {station_mse:.4f} | 测试集 R²: {station_r2:.4f}")

        # 图表 1: 滑动平均趋势图 (保持不变，仅修复黑底)
        window_size = 12
        all_targets_smooth = pd.Series(plot_targets).rolling(window=window_size, min_periods=1).mean().values
        all_preds_smooth = pd.Series(plot_preds).rolling(window=window_size, min_periods=1).mean().values

        plt.figure(figsize=(15, 6))
        plt.plot(plot_times, plot_targets, color='royalblue', linewidth=0.5, alpha=0.2, label='Actual (Raw)')
        plt.plot(plot_times, plot_preds, color='crimson', linewidth=0.5, alpha=0.2, label='Predicted (Raw)')
        plt.plot(plot_times, all_targets_smooth, label=f'Actual GPP (MA-{window_size})', color='royalblue', linewidth=1.5, alpha=0.9)
        plt.plot(plot_times, all_preds_smooth, label=f'Predicted GPP (MA-{window_size})', color='crimson', linewidth=1.5, linestyle='--', alpha=0.9)

        plt.title(f'[{station_name}] Test Set GPP Prediction (Moving Average Window={window_size})', fontsize=14, fontname='Arial')
        plt.xlabel('Time', fontsize=12, fontname='Arial')
        plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.grid(True, which='both', linestyle='--', alpha=0.5)
        plt.tight_layout()
        # 强制指定白色背景，关闭透明度
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_trend_ma.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # 图表 2: 局部放大图 (30天) —— 修改为使用 Raw Data
        zoom_days = 30
        steps_per_day = 24
        zoom_steps = zoom_days * steps_per_day
        zoom_steps = min(zoom_steps, len(plot_times))

        if zoom_steps > 0:
            peak_idx = np.argmax(all_targets_smooth)
            start_idx = max(0, peak_idx - zoom_steps // 2)
            end_idx = min(len(plot_times), start_idx + zoom_steps)

            if end_idx - start_idx < zoom_steps:
                start_idx = max(0, end_idx - zoom_steps)

            plt.figure(figsize=(15, 5))
            # 修改：此处换为 plot_targets 和 plot_preds（原始序列），以展示细节和高频变化
            plt.plot(plot_times[start_idx:end_idx], plot_targets[start_idx:end_idx],
                     label='Actual GPP (Raw)', color='royalblue', linewidth=1.5)
            plt.plot(plot_times[start_idx:end_idx], plot_preds[start_idx:end_idx],
                     label='Predicted GPP (Raw)', color='crimson', linewidth=1.5, linestyle='--', alpha=0.8)

            peak_date_str = pd.to_datetime(plot_times.iloc[peak_idx] if isinstance(plot_times, pd.Series) else plot_times[peak_idx]).strftime('%Y-%m-%d')

            plt.title(f'[{station_name}] 30-Day Zoomed Prediction (Raw Data Detail around {peak_date_str})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'})

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.6)
            plt.xticks(rotation=30)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_zoom_30days.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

        # 图表 3: 散点图
        plt.figure(figsize=(6, 6))
        plt.scatter(plot_targets, plot_preds, alpha=0.6, color='teal', s=15, edgecolors='k', linewidth=0.2)

        min_val = min(plot_targets.min(), plot_preds.min())
        max_val = max(plot_targets.max(), plot_preds.max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')

        plt.title(f'[{station_name}] Actual vs Predicted Scatter', fontname='Arial')
        plt.xlabel('Actual GPP', fontname='Arial')
        plt.ylabel('Predicted GPP', fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_scatter.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # ==========================================
        # 新增: 图表 4: 抽取任一年的数据展示时间序列
        # ==========================================
        years = plot_times.dt.year if hasattr(plot_times, 'dt') else pd.Series(plot_times).dt.year
        unique_years = years.unique()

        if len(unique_years) > 0:
            # 智能选择策略：为了图表丰满，默认选择数据点最多的一年，避免选到跨年只剩几天的数据
            target_year = years.value_counts().idxmax()
            year_mask = (years == target_year).values

            y_times = plot_times[year_mask]
            y_targets = plot_targets[year_mask]
            y_preds = plot_preds[year_mask]

            plt.figure(figsize=(15, 5))
            plt.plot(y_times, y_targets, label=f'Actual GPP ({target_year})', color='royalblue', linewidth=1.2, alpha=0.8)
            plt.plot(y_times, y_preds, label=f'Predicted GPP ({target_year})', color='crimson', linewidth=1.2, linestyle='--', alpha=0.8)

            plt.title(f'[{station_name}] Full Year GPP Time Series ({target_year})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'}, loc='upper right')

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.4)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_singleyear_{target_year}.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()
    # ------------------------------------------
    # 7. 全局指标汇总与 SHAP 分析
    # ------------------------------------------
    if len(global_all_targets) > 0:
        global_all_preds = np.array(global_all_preds)
        global_all_targets = np.array(global_all_targets)

        global_mse = np.mean((global_all_preds - global_all_targets)**2)
        global_r2 = r2_score(global_all_targets, global_all_preds)

        print("\n" + "="*50)
        print("🌎 所有站点测试集全局评估结果")
        print("="*50)
        print(f"有效总测试样本数 (剔除 GPP<0): {len(global_all_targets)}")
        print(f"Global Test MSE: {global_mse:.4f}")
        print(f"Global Test R² Score: {global_r2:.4f}")
        print("="*50)

        try:
            perform_shap_analysis(model, val_loader, device, output_img_folder, forcing_cols, state_cols)
        except Exception as e:
            pass

if __name__ == "__main__":
    train_time_aware_model()
#改为了局部归一化

📁 共找到 74 个站点文件。
   -> 训练集: 54 | 验证集: 10 | 测试集: 10

🔄 检测到本地进度，尝试恢复...
✅ 从第 21 轮恢复，历史最佳 MSE: 17.6594

🎯 开始测试集评估...
📢 站点: US-A39_CRO_Merged | 测试集 MSE: 10.2583 | 测试集 R²: 0.6157
📢 站点: US-Bar_DBF_Merged | 测试集 MSE: 6.8770 | 测试集 R²: 0.8466
📢 站点: US-HB3_ENF_Merged | 测试集 MSE: 3.4749 | 测试集 R²: 0.9115
📢 站点: US-SRM_WSA_Merged | 测试集 MSE: 2.2805 | 测试集 R²: 0.6206
📢 站点: US-xCP_GRA_Merged | 测试集 MSE: 1.1790 | 测试集 R²: 0.6948
📢 站点: US-xJR_OSH_Merged | 测试集 MSE: 1.0058 | 测试集 R²: 0.2564
📢 站点: US-xKZ_GRA_Merged | 测试集 MSE: 7.1762 | 测试集 R²: 0.8474
📢 站点: US-xNW_ENF_Merged | 测试集 MSE: 2.0244 | 测试集 R²: 0.4866
📢 站点: US-xST_DBF_Merged | 测试集 MSE: 5.3915 | 测试集 R²: 0.8918
📢 站点: US-xTL_WET_Merged | 测试集 MSE: 0.7174 | 测试集 R²: 0.6051

🌎 所有站点测试集全局评估结果
有效总测试样本数 (剔除 GPP<0): 428550
Global Test MSE: 3.5605
Global Test R² Score: 0.8481

🔍 开始执行 SHAP 变量重要性分析...


C:\Users\admin\AppData\Local\Temp\ipykernel_7160\2720692882.py:296: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
C:\Users\admin\AppData\Local\Temp\ipykernel_7160\2720692882.py:303: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)


✅ SHAP 生成完成，保存至: C:\Users\Admin\Desktop\实验三\全波段全变量DT\Results_TCN_Transformerkua分地类


In [3]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import shap
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score

# ==========================================
# 1. 数据集定义 (缺失值自动填补 + 断层检测)
# ==========================================
class TimeAwareMultiStationDataset(Dataset):
    def __init__(self, filepaths, seq_len=24, target_col='GPP_DT_VUT_REF', time_col='date',
                 forcing_cols=None, state_cols=None,
                 static_cols=['Lat', 'Long'], lc_col='Veg_ID',
                 feat_min_f=None, feat_max_f=None, feat_min_s=None, feat_max_s=None,
                 static_min=None, static_max=None, split_type='train'):

        self.seq_len = seq_len
        self.samples = []
        self.station_forcing, self.station_state, self.station_time_features = [], [], []
        self.station_targets, self.station_dates = [], []
        self.station_static, self.station_lc = [], []

        for filepath in filepaths:
            data = pd.read_csv(filepath)
            if time_col not in data.columns:
                print(f"⚠️ 在文件 {filepath} 中找不到时间列 '{time_col}'，跳过。")
                continue

            # 异常值清洗与插值填补 NaN
            data = data.replace([-9999, -9999.0, -999], np.nan)
            cols_to_clean = forcing_cols + state_cols + [target_col]
            cols_exist = [c for c in cols_to_clean if c in data.columns]

            data[cols_exist] = data[cols_exist].interpolate(method='linear', limit_direction='both')
            data = data.dropna(subset=cols_exist).reset_index(drop=True)

            data[time_col] = pd.to_datetime(data[time_col])
            data = data.sort_values(by=time_col).reset_index(drop=True)

            if len(data) < seq_len:
                continue

            dates = data[time_col]
            self.station_dates.append(dates.values)

            hours = dates.dt.hour.values
            days = dates.dt.dayofyear.values
            time_feats = np.column_stack([
                np.sin(2 * np.pi * hours / 24.0), np.cos(2 * np.pi * hours / 24.0),
                np.sin(2 * np.pi * days / 365.25), np.cos(2 * np.pi * days / 365.25)
            ])

            forcing_data = data[forcing_cols].values
            state_data = data[state_cols].values
            static_data = data[static_cols].values
            lc_data = data[lc_col].values.astype(np.int64)

            targets = data[target_col].values if target_col in data.columns else data.iloc[:, -1].values

            self.station_forcing.append(forcing_data)
            self.station_state.append(state_data)
            self.station_time_features.append(time_feats)
            self.station_targets.append(targets)
            self.station_static.append(static_data)
            self.station_lc.append(lc_data)

        if not self.station_forcing:
            print(f"⚠️ 警告：加载 {split_type} 数据为空。")
            return

        all_forcing_concat = np.vstack(self.station_forcing)
        all_state_concat = np.vstack(self.station_state)
        all_static_concat = np.vstack(self.station_static)

        self.feat_min_f = np.min(all_forcing_concat, axis=0) if feat_min_f is None else feat_min_f
        self.feat_max_f = np.max(all_forcing_concat, axis=0) if feat_max_f is None else feat_max_f
        self.feat_min_s = np.min(all_state_concat, axis=0) if feat_min_s is None else feat_min_s
        self.feat_max_s = np.max(all_state_concat, axis=0) if feat_max_s is None else feat_max_s
        self.static_min = np.min(all_static_concat, axis=0) if static_min is None else static_min
        self.static_max = np.max(all_static_concat, axis=0) if static_max is None else static_max

        # 归一化与样本切分
        for i in range(len(self.station_forcing)):
            self.station_forcing[i] = (self.station_forcing[i] - self.feat_min_f) / (self.feat_max_f - self.feat_min_f + 1e-8)
            self.station_state[i] = (self.station_state[i] - self.feat_min_s) / (self.feat_max_s - self.feat_min_s + 1e-8)
            self.station_static[i] = (self.station_static[i] - self.static_min) / (self.static_max - self.static_min + 1e-8)

            dates_array = self.station_dates[i]
            if len(dates_array) > 1:
                diffs = pd.Series(dates_array).diff()
                mode_step = diffs.mode()[0]
                expected_duration = mode_step * (self.seq_len - 1)

                num_samples = len(self.station_forcing[i]) - self.seq_len + 1
                if num_samples > 0:
                    for j in range(num_samples):
                        actual_duration = pd.Timedelta(dates_array[j + self.seq_len - 1] - dates_array[j])
                        if actual_duration == expected_duration:
                            self.samples.append((i, j))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        station_idx, start_idx = self.samples[idx]
        x_forcing = self.station_forcing[station_idx][start_idx : start_idx + self.seq_len]
        x_state = self.station_state[station_idx][start_idx : start_idx + self.seq_len]
        time_x = self.station_time_features[station_idx][start_idx : start_idx + self.seq_len]
        y = self.station_targets[station_idx][start_idx + self.seq_len - 1]
        target_date = self.station_dates[station_idx][start_idx + self.seq_len - 1]
        x_static = self.station_static[station_idx][start_idx : start_idx + self.seq_len]
        x_lc = self.station_lc[station_idx][start_idx : start_idx + self.seq_len]

        return (
            torch.tensor(x_forcing, dtype=torch.float32),
            torch.tensor(x_state, dtype=torch.float32),
            torch.tensor(time_x, dtype=torch.float32),
            torch.tensor(x_static, dtype=torch.float32),
            torch.tensor(x_lc, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            str(target_date)
        )

# ==========================================
# 2. TCN 基础模块定义
# ==========================================
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1, self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

# ==========================================
# 3. 交叉注意力融合模型
# ==========================================
class TCN_Transformer_CrossAttention(nn.Module):
    def __init__(self, num_forcing_features, num_state_features, seq_len,
                 num_static=2, num_lc_classes=10, lc_embed_dim=8,
                 d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super(TCN_Transformer_CrossAttention, self).__init__()

        self.tcn = TemporalConvNet(num_inputs=num_forcing_features, num_channels=[d_model] * 6, kernel_size=3, dropout=dropout)
        self.lc_embedding = nn.Embedding(num_embeddings=num_lc_classes, embedding_dim=lc_embed_dim)

        combined_state_dim = num_state_features + num_static + lc_embed_dim
        self.state_linear = nn.Linear(combined_state_dim, d_model)
        self.time_projector = nn.Linear(4, d_model)

        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.cross_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4), nn.ReLU(),
            nn.Linear(d_model // 4, 1)
        )

    def forward(self, x_forcing, x_state, time_x, x_static, x_lc):
        x_tcn_in = x_forcing.transpose(1, 2)
        f_tcn = self.tcn(x_tcn_in)
        f_met_memory = f_tcn.transpose(1, 2)

        lc_emb = self.lc_embedding(x_lc)
        combined_state = torch.cat([x_state, x_static, lc_emb], dim=-1)

        x_s_emb = self.state_linear(combined_state)
        time_emb = self.time_projector(time_x)
        x_state_combined = x_s_emb + time_emb
        f_state_global = self.transformer_encoder(x_state_combined)

        fused_features, _ = self.cross_attention(query=f_state_global, key=f_met_memory, value=f_met_memory)
        last_step_features = fused_features[:, -1, :]
        out = self.regressor(last_step_features)
        return out.squeeze(-1)

# ==========================================
# 4. SHAP 分析代码
# ==========================================
def perform_shap_analysis(model, dataloader, device, output_img_folder, forcing_cols, state_cols):
    print("\n🔍 开始执行 SHAP 变量重要性分析 (基于最佳Fold)...")
    model.eval()

    shap_loader = DataLoader(dataloader.dataset, batch_size=4000, shuffle=True)
    batch = next(iter(shap_loader))

    batch_forcing, batch_state, batch_time = batch[0].to(device), batch[1].to(device), batch[2].to(device)
    batch_static, batch_lc = batch[3].to(device), batch[4].to(device)

    bg_size, test_size = 1000, 1000
    if batch_forcing.size(0) < (bg_size + test_size):
        print("⚠️ 验证集 Batch size 太小，提供的数据量不足以支撑可靠的 SHAP 分析。")
        return

    bg_f, bg_s = batch_forcing[:bg_size], batch_state[:bg_size]
    test_f, test_s = batch_forcing[bg_size:bg_size+test_size], batch_state[bg_size:bg_size+test_size]

    class SHAPWrapper(nn.Module):
        def __init__(self, base_model, t_ref, st_ref, lc_ref):
            super().__init__()
            self.base_model = base_model
            self.t_ref = t_ref[:1]
            self.st_ref = st_ref[:1]
            self.lc_ref = lc_ref[:1]

        def forward(self, x_f, x_s):
            b_size = x_f.size(0)
            t_x = self.t_ref.expand(b_size, -1, -1)
            x_st = self.st_ref.expand(b_size, -1, -1)
            x_l = self.lc_ref.expand(b_size, -1)
            out = self.base_model(x_f, x_s, t_x, x_st, x_l)
            return out.unsqueeze(-1)

    wrapper_model = SHAPWrapper(model, batch_time, batch_static, batch_lc).to(device)
    wrapper_model.eval()

    explainer = shap.GradientExplainer(wrapper_model, [bg_f, bg_s])
    shap_values = explainer.shap_values([test_f, test_s])

    if isinstance(shap_values, list) and len(shap_values) == 1 and isinstance(shap_values[0], list):
        shap_values = shap_values[0]

    shap_forcing = np.array(shap_values[0])
    shap_state = np.array(shap_values[1])

    shap_forcing_2d = shap_forcing.reshape(-1, len(forcing_cols))
    shap_state_2d = shap_state.reshape(-1, len(state_cols))

    test_f_2d = test_f.cpu().numpy().reshape(-1, len(forcing_cols))
    test_s_2d = test_s.cpu().numpy().reshape(-1, len(state_cols))

    shap_combined = np.concatenate([shap_forcing_2d, shap_state_2d], axis=1)
    features_combined = np.concatenate([test_f_2d, test_s_2d], axis=1)
    feature_names = forcing_cols + state_cols

    os.makedirs(output_img_folder, exist_ok=True)

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
    plt.title("SHAP Summary: Global Feature Impact on GPP", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Summary_Plot.png"), dpi=300, facecolor='white')
    plt.close()

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)
    plt.title("SHAP Global Feature Importance (Magnitude)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Bar_Plot.png"), dpi=300, facecolor='white')
    plt.close()

    print(f"✅ SHAP 生成完成，保存至: {output_img_folder}")

# ==========================================
# 5. 主流程: 空间分层交叉验证 (Stratified CV)
# ==========================================
def train_stratified_cv_time_aware_model():
    seq_len = 96
    batch_size = 64
    num_epochs = 100
    learning_rate = 0.001
    patience = 10
    TIME_COLUMN_NAME = 'date'
    n_splits = 5 # 5折分层交叉验证

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data_folder = r"C:\Users\Admin\Desktop\实验三\全波段全变量DT"
    site_metadata_path = r"C:\Users\admin\Desktop\实验三\Successful_Sites_Filtered.csv"

    forcing_cols = ['SW_IN_F', 'SW_IN_POT', 'CO2_F_MDS', 'P_F', 'VPD_F', 'TA_F', 'TS_F_MDS_1', 'SWC_F_MDS_1', 'WS_F']
    state_cols = ['EPIC_Available_Mask', 'Band317nm_Ref', 'Band325nm_Ref', 'Band340nm_Ref',
                  'Band388nm_Ref', 'Band443nm_Ref', 'Band551nm_Ref', 'Band680nm_Ref',
                  'Band688nm_Ref', 'Band764nm_Ref', 'Band780nm_Ref', 'Mean_SZA', 'Mean_VZA', 'Mean_RAA']
    static_cols = ['Lat', 'Long']
    lc_col = 'Veg_ID'

    # 读取 CSV 文件进行站点与地类映射
    df_meta = pd.read_csv(site_metadata_path)
    site_to_veg = dict(zip(df_meta['Site'], df_meta['Veg']))

    all_files = glob.glob(os.path.join(data_folder, "*.[cC][sS][vV]"))
    if len(all_files) == 0:
        print("❌ 错误：未在指定目录找到特征CSV文件。")
        return

    file_list, veg_labels = [], []
    for f in all_files:
        fname = os.path.basename(f)
        matched_site = next((site for site in site_to_veg.keys() if site in fname), None)
        if matched_site:
            file_list.append(f)
            veg_labels.append(site_to_veg[matched_site])

    file_list = np.array(file_list)
    veg_labels = np.array(veg_labels)

    print(f"📁 成功匹配了 {len(file_list)} 个站点文件进行 {n_splits} 折分层交叉验证。")

    # 处理稀有地类避免 StratifiedKFold 报错
    unique_vegs, counts = np.unique(veg_labels, return_counts=True)
    rare_vegs = unique_vegs[counts < n_splits]
    stratify_labels = veg_labels.astype(object)
    for rare_veg in rare_vegs:
        stratify_labels[stratify_labels == rare_veg] = 'Other'

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    global_all_preds, global_all_targets = [], []
    cv_mse_scores, cv_r2_scores = [], []

    # ==========================================
    # ---> 新增：用于收集所有单一站点的测试结果 <---
    # ==========================================
    all_station_metrics = []

    # 用于追踪最佳模型的变量
    best_fold_overall_mse = float('inf')
    best_model_path = ""
    best_val_dataset = None

    # ==========================================
    # 开始 5-Fold 循环
    # ==========================================
    for fold, (train_val_idx, test_idx) in enumerate(skf.split(file_list, stratify_labels)):
        print(f"\n" + "="*50)
        print(f"🚀 开始 Fold {fold + 1} / {n_splits}")
        print("="*50)

        fold_output_folder = os.path.join(data_folder, f"Results_CV_Fold_{fold+1}")
        os.makedirs(fold_output_folder, exist_ok=True)

        test_files = file_list[test_idx].tolist()
        train_val_files = file_list[train_val_idx]

        # 划分训练集和早停验证集
        val_size = max(1, int(len(train_val_files) * 0.15))
        train_files = train_val_files[:-val_size].tolist()
        val_files = train_val_files[-val_size:].tolist()

        train_dataset = TimeAwareMultiStationDataset(
            train_files, seq_len, time_col=TIME_COLUMN_NAME, forcing_cols=forcing_cols,
            state_cols=state_cols, static_cols=static_cols, lc_col=lc_col, split_type=f'train_fold{fold+1}'
        )

        f_min_f, f_max_f = train_dataset.feat_min_f, train_dataset.feat_max_f
        f_min_s, f_max_s = train_dataset.feat_min_s, train_dataset.feat_max_s
        s_min, s_max = train_dataset.static_min, train_dataset.static_max

        val_dataset = TimeAwareMultiStationDataset(
            val_files, seq_len, time_col=TIME_COLUMN_NAME, forcing_cols=forcing_cols,
            state_cols=state_cols, static_cols=static_cols, lc_col=lc_col,
            feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
            static_min=s_min, static_max=s_max, split_type=f'val_fold{fold+1}'
        )

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = TCN_Transformer_CrossAttention(
            num_forcing_features=len(forcing_cols), num_state_features=len(state_cols),
            seq_len=seq_len, num_static=len(static_cols), num_lc_classes=10, lc_embed_dim=8
        ).to(device)

        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        checkpoint_best_path = os.path.join(fold_output_folder, f"checkpoint_best_fold{fold+1}.pth")

        # ==========================================
        # 新增：检索并加载已有模型参数（断点续训/权重复用）
        # ==========================================
        if os.path.exists(checkpoint_best_path):
            print(f"🔄 检测到已保存的模型权重: {os.path.basename(checkpoint_best_path)}")
            try:
                model.load_state_dict(torch.load(checkpoint_best_path, map_location=device))
                print("✅ 模型参数加载成功！将在此基础上继续训练。")
            except Exception as e:
                print(f"⚠️ 模型参数加载失败，将重新初始化训练。原因: {e}")

        # ---------------- 训练与早停 ----------------
        best_val_loss = float('inf')
        epochs_no_improve = 0

        for epoch in range(num_epochs):
            model.train()
            train_loss = 0
            for batch in train_loader:
                batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y = [b.to(device) for b in batch[:-1]]
                optimizer.zero_grad()
                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                loss = criterion(outputs, batch_y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch in val_loader:
                    batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y = [b.to(device) for b in batch[:-1]]
                    outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                    val_loss += criterion(outputs, batch_y).item()

            avg_val_loss = val_loss / len(val_loader)
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), checkpoint_best_path)
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f"🛑 Fold {fold+1} 早停于 Epoch {epoch+1}")
                break

        # ==========================================
        # 6. 当前 Fold 内部独立站点测试及绘制4张图
        # ==========================================
        model.load_state_dict(torch.load(checkpoint_best_path, map_location=device))
        model.eval()

        fold_preds, fold_targets = [], []

        for test_file in test_files:
            station_name = os.path.basename(test_file).replace('.csv', '')
            single_test_dataset = TimeAwareMultiStationDataset(
                [test_file], seq_len, time_col=TIME_COLUMN_NAME, forcing_cols=forcing_cols,
                state_cols=state_cols, static_cols=static_cols, lc_col=lc_col,
                feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
                static_min=s_min, static_max=s_max, split_type='test'
            )
            if len(single_test_dataset) == 0: continue

            single_test_loader = DataLoader(single_test_dataset, batch_size=batch_size, shuffle=False)
            all_preds, all_targets, all_times = [], [], []

            with torch.no_grad():
                for batch in single_test_loader:
                    batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, batch_dt = [b.to(device) if isinstance(b, torch.Tensor) else b for b in batch]
                    outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                    all_preds.extend(outputs.cpu().numpy())
                    all_targets.extend(batch_y.cpu().numpy())
                    all_times.extend(batch_dt)

            all_preds, all_targets = np.array(all_preds), np.array(all_targets)
            all_times = pd.to_datetime(all_times)

            # --- 严格数据清洗：剔除 GPP < 0 ---
            valid_mask = all_targets >= 0
            plot_targets = all_targets[valid_mask]
            plot_preds = all_preds[valid_mask]
            plot_times = all_times[valid_mask]

            if len(plot_targets) == 0: continue

            fold_preds.extend(plot_preds)
            fold_targets.extend(plot_targets)
            global_all_preds.extend(plot_preds)
            global_all_targets.extend(plot_targets)

            # ==========================================
            # ---> 新增：计算并输出当前单一站点的 R2 和 MSE <---
            # ==========================================
            if len(plot_targets) > 1:
                station_mse = np.mean((plot_preds - plot_targets)**2)
                station_r2 = r2_score(plot_targets, plot_preds)
                print(f"   📊 [Fold {fold+1} | 站点 {station_name}] 测试样本: {len(plot_targets)} | MSE: {station_mse:.4f} | R²: {station_r2:.4f}")

                # 记录到全局列表，方便后续保存为 CSV
                all_station_metrics.append({
                    'Fold': fold + 1,
                    'Station': station_name,
                    'Sample_Size': len(plot_targets),
                    'MSE': station_mse,
                    'R2': station_r2
                })
            else:
                print(f"   ⚠️ [Fold {fold+1} | 站点 {station_name}] 样本数过少，跳过指标计算。")

            # ---------------------------------------------
            # 图表 1: 滑动平均趋势图
            # ---------------------------------------------
            window_size = 12
            all_targets_smooth = pd.Series(plot_targets).rolling(window=window_size, min_periods=1).mean().values
            all_preds_smooth = pd.Series(plot_preds).rolling(window=window_size, min_periods=1).mean().values

            plt.figure(figsize=(15, 6))
            plt.plot(plot_times, plot_targets, color='royalblue', linewidth=0.5, alpha=0.2, label='Actual (Raw)')
            plt.plot(plot_times, plot_preds, color='crimson', linewidth=0.5, alpha=0.2, label='Predicted (Raw)')
            plt.plot(plot_times, all_targets_smooth, label=f'Actual GPP (MA-{window_size})', color='royalblue', linewidth=1.5, alpha=0.9)
            plt.plot(plot_times, all_preds_smooth, label=f'Predicted GPP (MA-{window_size})', color='crimson', linewidth=1.5, linestyle='--', alpha=0.9)

            plt.title(f'[{station_name}] Test Set GPP Prediction (Moving Average Window={window_size})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'})

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.5)
            plt.tight_layout()
            plt.savefig(os.path.join(fold_output_folder, f"{station_name}_trend_ma.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

            # ---------------------------------------------
            # 图表 2: 局部放大图 (30天 Raw Data)
            # ---------------------------------------------
            zoom_days = 30
            steps_per_day = 24
            zoom_steps = zoom_days * steps_per_day
            zoom_steps = min(zoom_steps, len(plot_times))

            if zoom_steps > 0:
                peak_idx = np.argmax(all_targets_smooth)
                start_idx = max(0, peak_idx - zoom_steps // 2)
                end_idx = min(len(plot_times), start_idx + zoom_steps)

                if end_idx - start_idx < zoom_steps:
                    start_idx = max(0, end_idx - zoom_steps)

                plt.figure(figsize=(15, 5))
                plt.plot(plot_times[start_idx:end_idx], plot_targets[start_idx:end_idx], label='Actual GPP (Raw)', color='royalblue', linewidth=1.5)
                plt.plot(plot_times[start_idx:end_idx], plot_preds[start_idx:end_idx], label='Predicted GPP (Raw)', color='crimson', linewidth=1.5, linestyle='--', alpha=0.8)

                peak_date_str = pd.to_datetime(plot_times.iloc[peak_idx] if isinstance(plot_times, pd.Series) else plot_times[peak_idx]).strftime('%Y-%m-%d')
                plt.title(f'[{station_name}] 30-Day Zoomed Prediction (Raw Data Detail around {peak_date_str})', fontsize=14, fontname='Arial')
                plt.xlabel('Time', fontsize=12, fontname='Arial')
                plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
                plt.legend(prop={'family': 'Arial'})

                ax = plt.gca()
                ax.tick_params(direction='in')
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                plt.grid(True, which='both', linestyle='--', alpha=0.6)
                plt.xticks(rotation=30)
                plt.tight_layout()
                plt.savefig(os.path.join(fold_output_folder, f"{station_name}_zoom_30days.png"), dpi=300, facecolor='white', transparent=False)
                plt.close()

            # ---------------------------------------------
            # 图表 3: 散点图
            # ---------------------------------------------
            plt.figure(figsize=(6, 6))
            plt.scatter(plot_targets, plot_preds, alpha=0.6, color='teal', s=15, edgecolors='k', linewidth=0.2)

            min_val = min(plot_targets.min(), plot_preds.min())
            max_val = max(plot_targets.max(), plot_preds.max())
            plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')

            plt.title(f'[{station_name}] Actual vs Predicted Scatter', fontname='Arial')
            plt.xlabel('Actual GPP', fontname='Arial')
            plt.ylabel('Predicted GPP', fontname='Arial')
            plt.legend(prop={'family': 'Arial'})

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            plt.savefig(os.path.join(fold_output_folder, f"{station_name}_scatter.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

            # ---------------------------------------------
            # 图表 4: 抽取任一年的数据展示完整时间序列
            # ---------------------------------------------
            years = plot_times.dt.year if hasattr(plot_times, 'dt') else pd.Series(plot_times).dt.year
            unique_years = years.unique()

            if len(unique_years) > 0:
                target_year = years.value_counts().idxmax()
                year_mask = (years == target_year).values

                y_times = plot_times[year_mask]
                y_targets = plot_targets[year_mask]
                y_preds = plot_preds[year_mask]

                plt.figure(figsize=(15, 5))
                plt.plot(y_times, y_targets, label=f'Actual GPP ({target_year})', color='royalblue', linewidth=1.2, alpha=0.8)
                plt.plot(y_times, y_preds, label=f'Predicted GPP ({target_year})', color='crimson', linewidth=1.2, linestyle='--', alpha=0.8)

                plt.title(f'[{station_name}] Full Year GPP Time Series ({target_year})', fontsize=14, fontname='Arial')
                plt.xlabel('Time', fontsize=12, fontname='Arial')
                plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
                plt.legend(prop={'family': 'Arial'}, loc='upper right')

                ax = plt.gca()
                ax.tick_params(direction='in')
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                plt.grid(True, which='both', linestyle='--', alpha=0.4)
                plt.tight_layout()
                plt.savefig(os.path.join(fold_output_folder, f"{station_name}_singleyear_{target_year}.png"), dpi=300, facecolor='white', transparent=False)
                plt.close()

        # ---------------- 当前折度量追踪 ----------------
        fold_preds, fold_targets = np.array(fold_preds), np.array(fold_targets)
        if len(fold_targets) > 0:
            fold_mse = np.mean((fold_preds - fold_targets)**2)
            fold_r2 = r2_score(fold_targets, fold_preds)
            cv_mse_scores.append(fold_mse)
            cv_r2_scores.append(fold_r2)
            print(f"✅ Fold {fold+1} 独立盲测完成 | MSE: {fold_mse:.4f} | R²: {fold_r2:.4f}")

            # 追踪表现最好的一折，留作最后做 SHAP 分析
            if fold_mse < best_fold_overall_mse:
                best_fold_overall_mse = fold_mse
                best_model_path = checkpoint_best_path
                best_val_dataset = val_dataset
                print(f"   [!] 记录当前最佳Fold: Fold {fold+1}，供后续 SHAP 分析使用。")

    # ==========================================
    # 7. 全局汇总与大一统图表绘制 (所有Fold结束后)
    # ==========================================
    global_all_preds = np.array(global_all_preds)
    global_all_targets = np.array(global_all_targets)

    global_mse = np.mean((global_all_preds - global_all_targets)**2)
    global_r2 = r2_score(global_all_targets, global_all_preds)

    print("\n" + "🏆"*20)
    print(f"🌎 {n_splits}-Fold 分层空间交叉验证 最终全局盲测结果:")
    print(f"   参与总测试样本数: {len(global_all_targets)} (覆盖全部验证地类站点)")
    print(f"   Average Fold R²:  {np.mean(cv_r2_scores):.4f} ± {np.std(cv_r2_scores):.4f}")
    print(f"   Global Test MSE:  {global_mse:.4f}")
    print(f"   Global Test R² :  {global_r2:.4f}")
    print("🏆"*20)

    # ==========================================
    # ---> 新增：将所有单站点的评估结果保存为 CSV 表格 <---
    # ==========================================
    if all_station_metrics:
        df_metrics = pd.DataFrame(all_station_metrics)
        metrics_save_path = os.path.join(data_folder, "All_Stations_Metrics_Summary.csv")
        df_metrics.to_csv(metrics_save_path, index=False, encoding='utf-8-sig')
        print(f"\n📂 所有独立站点的评估指标（MSE, R²）已保存至: {metrics_save_path}")
        print("💡 提示：你可以直接将此 CSV 文件导入 Excel，用于制作论文中的多站点表现对比表。")

    # 绘制包含全量测试站点的权威散点图
    plt.figure(figsize=(8, 8))
    plt.scatter(global_all_targets, global_all_preds, alpha=0.15, color='teal', s=8, edgecolors='none')
    min_val = min(global_all_targets.min(), global_all_preds.min())
    max_val = max(global_all_targets.max(), global_all_preds.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')
    plt.title(f'Global Spatial CV: Actual vs Predicted GPP (Overall R²={global_r2:.3f})', fontname='Arial', fontsize=15)
    plt.xlabel('Actual GPP (All CV Test Sites Combined)', fontname='Arial', fontsize=13)
    plt.ylabel('Predicted GPP (All CV Test Sites Combined)', fontname='Arial', fontsize=13)
    plt.legend(prop={'family': 'Arial'})
    ax = plt.gca()
    ax.tick_params(direction='in')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(data_folder, "Global_CV_All_Sites_Scatter.png"), dpi=600, facecolor='white')
    plt.close()

    # ==========================================
    # 8. 基于最佳 Fold 的权威 SHAP 分析
    # ==========================================
    if best_model_path and best_val_dataset is not None:
        shap_output_folder = os.path.join(data_folder, "SHAP_Analysis_BestFold")
        print(f"\n💡 正在加载表现最佳的模型 ({os.path.basename(best_model_path)}) 进行归因...")

        best_model = TCN_Transformer_CrossAttention(
            num_forcing_features=len(forcing_cols), num_state_features=len(state_cols),
            seq_len=seq_len, num_static=len(static_cols), num_lc_classes=10, lc_embed_dim=8
        ).to(device)
        best_model.load_state_dict(torch.load(best_model_path, map_location=device))

        best_val_loader = DataLoader(best_val_dataset, batch_size=batch_size, shuffle=False)
        perform_shap_analysis(best_model, best_val_loader, device, shap_output_folder, forcing_cols, state_cols)

if __name__ == "__main__":
    train_stratified_cv_time_aware_model()
#k折

📁 成功匹配了 74 个站点文件进行 5 折分层交叉验证。

🚀 开始 Fold 1 / 5
🔄 检测到已保存的模型权重: checkpoint_best_fold1.pth
✅ 模型参数加载成功！将在此基础上继续训练。
🛑 Fold 1 早停于 Epoch 15
   📊 [Fold 1 | 站点 CA-Cbo_DBF_Merged] 测试样本: 34898 | MSE: 21.3819 | R²: 0.7958
   📊 [Fold 1 | 站点 CA-Mer_WET_Merged] 测试样本: 34898 | MSE: 9.6576 | R²: 0.0622
   📊 [Fold 1 | 站点 CA-TP1_ENF_Merged] 测试样本: 61178 | MSE: 22.7025 | R²: 0.7381
   📊 [Fold 1 | 站点 CR-Fsc_CRO_Merged] 测试样本: 26209 | MSE: 125.9978 | R²: 0.3805
   📊 [Fold 1 | 站点 US-A37_CVM_Merged] 测试样本: 8683 | MSE: 11.0828 | R²: 0.6054
   📊 [Fold 1 | 站点 US-Bar_DBF_Merged] 测试样本: 61178 | MSE: 8.3376 | R²: 0.8140
   📊 [Fold 1 | 站点 US-PAS_GRA_Merged] 测试样本: 17420 | MSE: 58.5491 | R²: 0.4828
   📊 [Fold 1 | 站点 US-SP1_ENF_Merged] 测试样本: 34898 | MSE: 5.7257 | R²: 0.8543
   📊 [Fold 1 | 站点 US-UC2_CVM_Merged] 测试样本: 43753 | MSE: 12.4205 | R²: 0.8205
   📊 [Fold 1 | 站点 US-xCP_GRA_Merged] 测试样本: 43753 | MSE: 1.5726 | R²: 0.5930
   📊 [Fold 1 | 站点 US-xGR_DBF_Merged] 测试样本: 43753 | MSE: 15.6355 | R²: 0.8236
   📊 [Fold 1 | 站点 US-xMB

C:\Users\admin\AppData\Local\Temp\ipykernel_19964\2258308740.py:280: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
C:\Users\admin\AppData\Local\Temp\ipykernel_19964\2258308740.py:287: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)


✅ SHAP 生成完成，保存至: C:\Users\Admin\Desktop\实验三\全波段全变量DT\SHAP_Analysis_BestFold


In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import shap
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

# ==========================================
# 1. 数据集定义 (缺失值自动填补 + 断层检测)
# ==========================================
class TimeAwareMultiStationDataset(Dataset):
    def __init__(self, filepaths, seq_len=24, target_col='GPP_DT_VUT_REF', time_col='date',
                 forcing_cols=None, state_cols=None,
                 static_cols=['Lat', 'Long'], lc_col='Veg_ID',
                 feat_min_f=None, feat_max_f=None, feat_min_s=None, feat_max_s=None,
                 static_min=None, static_max=None, split_type='train'):

        self.seq_len = seq_len
        self.samples = []

        self.station_forcing = []
        self.station_state = []
        self.station_time_features = []
        self.station_targets = []
        self.station_dates = []

        self.station_static = []
        self.station_lc = []

        for filepath in filepaths:
            data = pd.read_csv(filepath)
            if time_col not in data.columns:
                print(f"⚠️ 在文件 {filepath} 中找不到时间列 '{time_col}'，跳过。")
                continue

            # 【修改后】异常值清洗：仅替换为 NaN 并直接剔除，取消线性插值
            data = data.replace([-9999, -9999.0, -999], np.nan)
            cols_to_clean = forcing_cols + state_cols + [target_col]
            cols_exist = [c for c in cols_to_clean if c in data.columns]

            # 必须保留 dropna：因为 PyTorch 无法计算 NaN，带 NaN 的行必须直接删掉
            data = data.dropna(subset=cols_exist).reset_index(drop=True)

            data[time_col] = pd.to_datetime(data[time_col])
            data = data.sort_values(by=time_col).reset_index(drop=True)

            if len(data) < seq_len:
                continue

            dates = data[time_col]
            self.station_dates.append(dates.values)

            hours = dates.dt.hour.values
            days = dates.dt.dayofyear.values
            time_feats = np.column_stack([
                np.sin(2 * np.pi * hours / 24.0), np.cos(2 * np.pi * hours / 24.0),
                np.sin(2 * np.pi * days / 365.25), np.cos(2 * np.pi * days / 365.25)
            ])

            forcing_data = data[forcing_cols].values
            state_data = data[state_cols].values
            static_data = data[static_cols].values
            lc_data = data[lc_col].values.astype(np.int64)

            if target_col in data.columns:
                targets = data[target_col].values
            else:
                targets = data.iloc[:, -1].values

            self.station_forcing.append(forcing_data)
            self.station_state.append(state_data)
            self.station_time_features.append(time_feats)
            self.station_targets.append(targets)
            self.station_static.append(static_data)
            self.station_lc.append(lc_data)

        if not self.station_forcing:
            raise ValueError(f"加载 {split_type} 数据失败，可能所有数据均被清洗掉或长度不足。")

        all_forcing_concat = np.vstack(self.station_forcing)
        all_state_concat = np.vstack(self.station_state)
        all_static_concat = np.vstack(self.station_static)

        self.feat_min_f = np.min(all_forcing_concat, axis=0) if feat_min_f is None else feat_min_f
        self.feat_max_f = np.max(all_forcing_concat, axis=0) if feat_max_f is None else feat_max_f
        self.feat_min_s = np.min(all_state_concat, axis=0) if feat_min_s is None else feat_min_s
        self.feat_max_s = np.max(all_state_concat, axis=0) if feat_max_s is None else feat_max_s

        self.static_min = np.min(all_static_concat, axis=0) if static_min is None else static_min
        self.static_max = np.max(all_static_concat, axis=0) if static_max is None else static_max

        # 归一化与样本切分 (包含时间连续性断层检测)
        for i in range(len(self.station_forcing)):
            self.station_forcing[i] = (self.station_forcing[i] - self.feat_min_f) / (self.feat_max_f - self.feat_min_f + 1e-8)
            self.station_state[i] = (self.station_state[i] - self.feat_min_s) / (self.feat_max_s - self.feat_min_s + 1e-8)
            self.station_static[i] = (self.station_static[i] - self.static_min) / (self.static_max - self.static_min + 1e-8)

            dates_array = self.station_dates[i]
            if len(dates_array) > 1:
                diffs = pd.Series(dates_array).diff()
                mode_step = diffs.mode()[0]
                expected_duration = mode_step * (self.seq_len - 1)

                num_samples = len(self.station_forcing[i]) - self.seq_len + 1
                if num_samples > 0:
                    for j in range(num_samples):
                        actual_duration = pd.Timedelta(dates_array[j + self.seq_len - 1] - dates_array[j])
                        if actual_duration == expected_duration:
                            self.samples.append((i, j))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        station_idx, start_idx = self.samples[idx]

        x_forcing = self.station_forcing[station_idx][start_idx : start_idx + self.seq_len]
        x_state = self.station_state[station_idx][start_idx : start_idx + self.seq_len]
        time_x = self.station_time_features[station_idx][start_idx : start_idx + self.seq_len]
        y = self.station_targets[station_idx][start_idx + self.seq_len - 1]
        target_date = self.station_dates[station_idx][start_idx + self.seq_len - 1]

        x_static = self.station_static[station_idx][start_idx : start_idx + self.seq_len]
        x_lc = self.station_lc[station_idx][start_idx : start_idx + self.seq_len]

        return (
            torch.tensor(x_forcing, dtype=torch.float32),
            torch.tensor(x_state, dtype=torch.float32),
            torch.tensor(time_x, dtype=torch.float32),
            torch.tensor(x_static, dtype=torch.float32),
            torch.tensor(x_lc, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            str(target_date)
        )

# ==========================================
# 2. TCN 基础模块定义
# ==========================================
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1, self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

# ==========================================
# 3. 交叉注意力融合模型
# ==========================================
class TCN_Transformer_CrossAttention(nn.Module):
    def __init__(self, num_forcing_features, num_state_features, seq_len,
                 num_static=2, num_lc_classes=13, lc_embed_dim=8,
                 d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super(TCN_Transformer_CrossAttention, self).__init__()

        self.tcn = TemporalConvNet(num_inputs=num_forcing_features,
                                   num_channels=[d_model] * 6,
                                   kernel_size=3, dropout=dropout)

        self.lc_embedding = nn.Embedding(num_embeddings=num_lc_classes, embedding_dim=lc_embed_dim)

        combined_state_dim = num_state_features + num_static + lc_embed_dim
        self.state_linear = nn.Linear(combined_state_dim, d_model)
        self.time_projector = nn.Linear(4, d_model)

        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.cross_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4), nn.ReLU(),
            nn.Linear(d_model // 4, 1)
        )

    def forward(self, x_forcing, x_state, time_x, x_static, x_lc):
        x_tcn_in = x_forcing.transpose(1, 2)
        f_tcn = self.tcn(x_tcn_in)
        f_met_memory = f_tcn.transpose(1, 2)

        lc_emb = self.lc_embedding(x_lc)
        combined_state = torch.cat([x_state, x_static, lc_emb], dim=-1)

        x_s_emb = self.state_linear(combined_state)
        time_emb = self.time_projector(time_x)
        x_state_combined = x_s_emb + time_emb
        f_state_global = self.transformer_encoder(x_state_combined)

        fused_features, _ = self.cross_attention(
            query=f_state_global,
            key=f_met_memory,
            value=f_met_memory
        )

        last_step_features = fused_features[:, -1, :]
        out = self.regressor(last_step_features)
        return out.squeeze(-1)

# ==========================================
# 4. SHAP 分析代码
# ==========================================
def perform_shap_analysis(model, dataloader, device, output_img_folder, forcing_cols, state_cols):
    print("\n🔍 开始执行 SHAP 变量重要性分析...")
    model.eval()

    shap_loader = DataLoader(dataloader.dataset, batch_size=4000, shuffle=True)
    batch = next(iter(shap_loader))

    batch_forcing, batch_state, batch_time = batch[0].to(device), batch[1].to(device), batch[2].to(device)
    batch_static, batch_lc = batch[3].to(device), batch[4].to(device)

    bg_size, test_size = 1000, 1000
    if batch_forcing.size(0) < (bg_size + test_size):
        print("⚠️ Batch size 太小，跳过 SHAP 分析。")
        return

    bg_f, bg_s = batch_forcing[:bg_size], batch_state[:bg_size]
    test_f, test_s = batch_forcing[bg_size:bg_size+test_size], batch_state[bg_size:bg_size+test_size]

    class SHAPWrapper(nn.Module):
        def __init__(self, base_model, t_ref, st_ref, lc_ref):
            super().__init__()
            self.base_model = base_model
            self.t_ref = t_ref[:1]
            self.st_ref = st_ref[:1]
            self.lc_ref = lc_ref[:1]

        def forward(self, x_f, x_s):
            b_size = x_f.size(0)
            t_x = self.t_ref.expand(b_size, -1, -1)
            x_st = self.st_ref.expand(b_size, -1, -1)
            x_l = self.lc_ref.expand(b_size, -1)
            out = self.base_model(x_f, x_s, t_x, x_st, x_l)
            return out.unsqueeze(-1)

    wrapper_model = SHAPWrapper(model, batch_time, batch_static, batch_lc).to(device)
    wrapper_model.eval()

    explainer = shap.GradientExplainer(wrapper_model, [bg_f, bg_s])
    shap_values = explainer.shap_values([test_f, test_s])

    if isinstance(shap_values, list) and len(shap_values) == 1 and isinstance(shap_values[0], list):
        shap_values = shap_values[0]

    shap_forcing = np.array(shap_values[0])
    shap_state = np.array(shap_values[1])

    shap_forcing_2d = shap_forcing.reshape(-1, len(forcing_cols))
    shap_state_2d = shap_state.reshape(-1, len(state_cols))

    test_f_2d = test_f.cpu().numpy().reshape(-1, len(forcing_cols))
    test_s_2d = test_s.cpu().numpy().reshape(-1, len(state_cols))

    shap_combined = np.concatenate([shap_forcing_2d, shap_state_2d], axis=1)
    features_combined = np.concatenate([test_f_2d, test_s_2d], axis=1)
    feature_names = forcing_cols + state_cols

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
    plt.title("SHAP Summary: Global Feature Impact on GPP (Time-Flattened)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Summary_Plot.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)
    plt.title("SHAP Global Feature Importance (Magnitude)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Bar_Plot.png"), dpi=300)
    plt.close()

    print(f"✅ SHAP 生成完成，保存至: {output_img_folder}")

# ==========================================
# 5. 主流程
# ==========================================
def train_time_aware_model():
    seq_len = 96
    batch_size = 64
    num_epochs = 100
    learning_rate = 0.001
    patience = 10
    TIME_COLUMN_NAME = 'date'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data_folder = r"C:\Users\Admin\Desktop\实验四\全波段全变量DT"
    output_img_folder = os.path.join(data_folder, "全149站点")
    os.makedirs(output_img_folder, exist_ok=True)

    forcing_cols = ['SW_IN_F', 'SW_IN_POT', 'CO2_F_MDS', 'P_F', 'VPD_F', 'TA_F', 'TS_F_MDS_1', 'SWC_F_MDS_1', 'WS_F']
    state_cols = ['EPIC_Available_Mask', 'Band317nm_Ref', 'Band325nm_Ref', 'Band340nm_Ref',
                  'Band388nm_Ref', 'Band443nm_Ref', 'Band551nm_Ref', 'Band680nm_Ref',
                  'Band688nm_Ref', 'Band764nm_Ref', 'Band780nm_Ref', 'Mean_SZA', 'Mean_VZA', 'Mean_RAA']
    static_cols = ['Lat', 'Long']
    lc_col = 'Veg_ID'

    val_sites = [
    'CR-Fsc_CRO', 'US-Mo1_CRO', 'FR-Aur_CRO', 'US-TKs_CSH', 'US-UC1_CVM',
    'CN-HeM_DBF', 'US-Rpf_DBF', 'KR-PcD_DBF', 'AU-Gin_EBF', 'US-xJE_ENF',
    'IL-Yat_ENF', 'DE-Har_ENF', 'FI-Ken_ENF', 'US-xKZ_GRA', 'IT-Tor_GRA',
    'US-Var_GRA', 'CH-Crs_GRA', 'KR-JjM_MF', 'US-xMB_OSH', 'ES-LM1_SAV',
    'JP-BBY_WET', 'AU-Cpr_WSA'
]

    test_sites = [
    'US-A39_CRO', 'CZ-KrP_CRO', 'AU-MvB_CRO', 'US-Rls_CSH', 'US-A37_CVM',
    'US-xUK_DBF', 'CZ-Lnz_DBF', 'FR-Fon_DBF', 'JP-Fhk_DNF', 'KR-WdE_EBF',
    'BE-Bra_ENF', 'US-xSB_ENF', 'US-MEF_ENF', 'US-xWR_ENF', 'AU-Sil_GRA',
    'BE-Dor_GRA', 'AU-Wel_GRA', 'US-KFS_GRA', 'CH-Lae_MF', 'DE-RuC_OSH',
    'US-xSJ_SAV', 'SE-Deg_WET', 'US-SRM_WSA'
    ]

    all_files = glob.glob(os.path.join(data_folder, "*.[cC][sS][vV]"))
    if not all_files:
        print("❌ 错误：未在指定目录找到CSV文件。")
        return

    train_files, val_files, test_files = [], [], []
    for f in all_files:
        fname = os.path.basename(f)
        if any(site in fname for site in test_sites):
            test_files.append(f)
        elif any(site in fname for site in val_sites):
            val_files.append(f)
        else:
            train_files.append(f)

    print(f"📁 共找到 {len(all_files)} 个站点文件。")
    print(f"   -> 训练集: {len(train_files)} | 验证集: {len(val_files)} | 测试集: {len(test_files)}")

    train_dataset = TimeAwareMultiStationDataset(
        train_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col, split_type='train'
    )

    f_min_f, f_max_f = train_dataset.feat_min_f, train_dataset.feat_max_f
    f_min_s, f_max_s = train_dataset.feat_min_s, train_dataset.feat_max_s
    s_min, s_max = train_dataset.static_min, train_dataset.static_max

    val_dataset = TimeAwareMultiStationDataset(
        val_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col,
        feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
        static_min=s_min, static_max=s_max, split_type='val'
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = TCN_Transformer_CrossAttention(
        num_forcing_features=len(forcing_cols),
        num_state_features=len(state_cols),
        seq_len=seq_len,
        num_static=len(static_cols),
        num_lc_classes=13,
        lc_embed_dim=8
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    checkpoint_latest_path = os.path.join(output_img_folder, "checkpoint_latest.pth")
    checkpoint_best_path = os.path.join(output_img_folder, "checkpoint_best.pth")

    start_epoch = 0
    best_val_loss = float('inf')
    epochs_no_improve = 0

    if os.path.exists(checkpoint_latest_path):
        print(f"\n🔄 检测到本地进度，尝试恢复...")
        try:
            checkpoint = torch.load(checkpoint_latest_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_loss = checkpoint['best_val_loss']
            epochs_no_improve = checkpoint['epochs_no_improve']
            print(f"✅ 从第 {start_epoch} 轮恢复，历史最佳 MSE: {best_val_loss:.4f}")
        except Exception as e:
            print(f"⚠️ 无法加载历史进度，将重新开始。错误信息: {e}")

    if start_epoch < num_epochs and epochs_no_improve < patience:
        print(f"🚀 开始训练...\n" + "-"*40)
        for epoch in range(start_epoch, num_epochs):
            model.train()
            train_loss = 0

            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in train_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                optimizer.zero_grad()
                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                loss = criterion(outputs, batch_y)
                loss.backward()

                # 梯度裁剪
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in val_loader:
                    batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                    batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                    outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            print(f"Epoch [{epoch+1:03d}/{num_epochs}] | Train MSE: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f}")

            if avg_val_loss < best_val_loss:
                print(f"   🌟 新的最佳模型！MSE: {avg_val_loss:.4f}。")
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), checkpoint_best_path)
            else:
                epochs_no_improve += 1
                print(f"   ⏳ 验证集未改善 (早停: {epochs_no_improve}/{patience})")

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'epochs_no_improve': epochs_no_improve
            }, checkpoint_latest_path)

            if epochs_no_improve >= patience:
                print(f"\n🛑 早停机制触发，训练结束。")
                break
            print("-" * 40)

# ==========================================
    # 6. 分站点独立测试评估 (整合高级绘图逻辑)
    # ==========================================
    if os.path.exists(checkpoint_best_path):
        print(f"\n🎯 开始测试集评估...")
        model.load_state_dict(torch.load(checkpoint_best_path, map_location=device))
    model.eval()

    global_all_preds, global_all_targets = [], []
    for test_file in test_files:
        station_name = os.path.basename(test_file).replace('.csv', '')
        single_test_dataset = TimeAwareMultiStationDataset(
            [test_file], seq_len, time_col=TIME_COLUMN_NAME,
            forcing_cols=forcing_cols, state_cols=state_cols,
            static_cols=static_cols, lc_col=lc_col,
            feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
            static_min=s_min, static_max=s_max, split_type='test'
        )

        if len(single_test_dataset) == 0: continue

        single_test_loader = DataLoader(single_test_dataset, batch_size=batch_size, shuffle=False)
        all_preds, all_targets, all_times = [], [], []

        with torch.no_grad():
            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, batch_dt in single_test_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc = batch_static.to(device), batch_lc.to(device)

                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(batch_y.numpy())
                all_times.extend(batch_dt)

        all_preds, all_targets = np.array(all_preds), np.array(all_targets)
        all_times = pd.to_datetime(all_times)

        # 应用数据清洗要求：剔除 GPP < 0 的异常值
        valid_mask = all_targets >= 0
        plot_targets = all_targets[valid_mask]
        plot_preds = all_preds[valid_mask]
        plot_times = all_times[valid_mask]

        if len(plot_targets) == 0:
            continue

        global_all_preds.extend(plot_preds)
        global_all_targets.extend(plot_targets)

        station_mse = np.mean((plot_preds - plot_targets)**2)
        station_r2 = r2_score(plot_targets, plot_preds)

        print(f"📢 站点: {station_name} | 测试集 MSE: {station_mse:.4f} | 测试集 R²: {station_r2:.4f}")

        # 图表 1: 滑动平均趋势图 (保持不变，仅修复黑底)
        window_size = 12
        all_targets_smooth = pd.Series(plot_targets).rolling(window=window_size, min_periods=1).mean().values
        all_preds_smooth = pd.Series(plot_preds).rolling(window=window_size, min_periods=1).mean().values

        plt.figure(figsize=(15, 6))
        plt.plot(plot_times, plot_targets, color='royalblue', linewidth=0.5, alpha=0.2, label='Actual (Raw)')
        plt.plot(plot_times, plot_preds, color='crimson', linewidth=0.5, alpha=0.2, label='Predicted (Raw)')
        plt.plot(plot_times, all_targets_smooth, label=f'Actual GPP (MA-{window_size})', color='royalblue', linewidth=1.5, alpha=0.9)
        plt.plot(plot_times, all_preds_smooth, label=f'Predicted GPP (MA-{window_size})', color='crimson', linewidth=1.5, linestyle='--', alpha=0.9)

        plt.title(f'[{station_name}] Test Set GPP Prediction (Moving Average Window={window_size})', fontsize=14, fontname='Arial')
        plt.xlabel('Time', fontsize=12, fontname='Arial')
        plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.grid(True, which='both', linestyle='--', alpha=0.5)
        plt.tight_layout()
        # 强制指定白色背景，关闭透明度
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_trend_ma.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # 图表 2: 局部放大图 (30天) —— 修改为使用 Raw Data
        zoom_days = 30
        steps_per_day = 24
        zoom_steps = zoom_days * steps_per_day
        zoom_steps = min(zoom_steps, len(plot_times))

        if zoom_steps > 0:
            peak_idx = np.argmax(all_targets_smooth)
            start_idx = max(0, peak_idx - zoom_steps // 2)
            end_idx = min(len(plot_times), start_idx + zoom_steps)

            if end_idx - start_idx < zoom_steps:
                start_idx = max(0, end_idx - zoom_steps)

            plt.figure(figsize=(15, 5))
            # 修改：此处换为 plot_targets 和 plot_preds（原始序列），以展示细节和高频变化
            plt.plot(plot_times[start_idx:end_idx], plot_targets[start_idx:end_idx],
                     label='Actual GPP (Raw)', color='royalblue', linewidth=1.5)
            plt.plot(plot_times[start_idx:end_idx], plot_preds[start_idx:end_idx],
                     label='Predicted GPP (Raw)', color='crimson', linewidth=1.5, linestyle='--', alpha=0.8)

            peak_date_str = pd.to_datetime(plot_times.iloc[peak_idx] if isinstance(plot_times, pd.Series) else plot_times[peak_idx]).strftime('%Y-%m-%d')

            plt.title(f'[{station_name}] 30-Day Zoomed Prediction (Raw Data Detail around {peak_date_str})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'})

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.6)
            plt.xticks(rotation=30)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_zoom_30days.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

        # 图表 3: 散点图
        plt.figure(figsize=(6, 6))
        plt.scatter(plot_targets, plot_preds, alpha=0.6, color='teal', s=15, edgecolors='k', linewidth=0.2)

        min_val = min(plot_targets.min(), plot_preds.min())
        max_val = max(plot_targets.max(), plot_preds.max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')

        plt.title(f'[{station_name}] Actual vs Predicted Scatter', fontname='Arial')
        plt.xlabel('Actual GPP', fontname='Arial')
        plt.ylabel('Predicted GPP', fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_scatter.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # ==========================================
        # 新增: 图表 4: 抽取任一年的数据展示时间序列
        # ==========================================
        years = plot_times.dt.year if hasattr(plot_times, 'dt') else pd.Series(plot_times).dt.year
        unique_years = years.unique()

        if len(unique_years) > 0:
            # 智能选择策略：为了图表丰满，默认选择数据点最多的一年，避免选到跨年只剩几天的数据
            target_year = years.value_counts().idxmax()
            year_mask = (years == target_year).values

            y_times = plot_times[year_mask]
            y_targets = plot_targets[year_mask]
            y_preds = plot_preds[year_mask]

            plt.figure(figsize=(15, 5))
            plt.plot(y_times, y_targets, label=f'Actual GPP ({target_year})', color='royalblue', linewidth=1.2, alpha=0.8)
            plt.plot(y_times, y_preds, label=f'Predicted GPP ({target_year})', color='crimson', linewidth=1.2, linestyle='--', alpha=0.8)

            plt.title(f'[{station_name}] Full Year GPP Time Series ({target_year})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'}, loc='upper right')

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.4)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_singleyear_{target_year}.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()
    # ------------------------------------------
    # 7. 全局指标汇总与 SHAP 分析
    # ------------------------------------------
    if len(global_all_targets) > 0:
        global_all_preds = np.array(global_all_preds)
        global_all_targets = np.array(global_all_targets)

        global_mse = np.mean((global_all_preds - global_all_targets)**2)
        global_r2 = r2_score(global_all_targets, global_all_preds)

        print("\n" + "="*50)
        print("🌎 所有站点测试集全局评估结果")
        print("="*50)
        print(f"有效总测试样本数 (剔除 GPP<0): {len(global_all_targets)}")
        print(f"Global Test MSE: {global_mse:.4f}")
        print(f"Global Test R² Score: {global_r2:.4f}")
        print("="*50)

        try:
            perform_shap_analysis(model, val_loader, device, output_img_folder, forcing_cols, state_cols)
        except Exception as e:
            pass

if __name__ == "__main__":
    train_time_aware_model()
#改为了局部归一化，目前最佳

D:\miniconda3\envs\dl_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📁 共找到 149 个站点文件。
   -> 训练集: 104 | 验证集: 22 | 测试集: 23

🔄 检测到本地进度，尝试恢复...
✅ 从第 18 轮恢复，历史最佳 MSE: 12.2771

🎯 开始测试集评估...
📢 站点: AU-MvB_CRO_Merged | 测试集 MSE: 52.4937 | 测试集 R²: 0.5704
📢 站点: AU-Sil_GRA_Merged | 测试集 MSE: 11.4889 | 测试集 R²: 0.5335
📢 站点: AU-Wel_GRA_Merged | 测试集 MSE: 9.3147 | 测试集 R²: 0.7597
📢 站点: BE-Bra_ENF_Merged | 测试集 MSE: 5.3792 | 测试集 R²: 0.8746
📢 站点: BE-Dor_GRA_Merged | 测试集 MSE: 12.7920 | 测试集 R²: 0.7894
📢 站点: CH-Lae_MF_Merged | 测试集 MSE: 16.0093 | 测试集 R²: 0.8221
📢 站点: CZ-KrP_CRO_Merged | 测试集 MSE: 14.9953 | 测试集 R²: 0.6716
📢 站点: CZ-Lnz_DBF_Merged | 测试集 MSE: 11.3071 | 测试集 R²: 0.8773
📢 站点: DE-RuC_OSH_Merged | 测试集 MSE: 2.9707 | 测试集 R²: 0.8590
📢 站点: FR-Fon_DBF_Merged | 测试集 MSE: 7.0458 | 测试集 R²: 0.9032
📢 站点: JP-Fhk_DNF_Merged | 测试集 MSE: 10.8666 | 测试集 R²: 0.8657
📢 站点: KR-WdE_EBF_Merged | 测试集 MSE: 33.2352 | 测试集 R²: 0.5308
📢 站点: SE-Deg_WET_Merged | 测试集 MSE: 0.7131 | 测试集 R²: 0.7135
📢 站点: US-A37_CVM_Merged | 测试集 MSE: 18.8738 | 测试集 R²: 0.3661
📢 站点: US-A39_CRO_Merged | 测试集 MSE: 23.9806 | 测试集 R²

C:\Users\admin\AppData\Local\Temp\ipykernel_2016\2214656726.py:296: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
C:\Users\admin\AppData\Local\Temp\ipykernel_2016\2214656726.py:303: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)


✅ SHAP 生成完成，保存至: C:\Users\Admin\Desktop\实验四\全波段全变量DT\全149站点


In [2]:
#各地类不超过8个站点
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import shap
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

# ==========================================
# 1. 数据集定义 (缺失值自动填补 + 断层检测)
# ==========================================
class TimeAwareMultiStationDataset(Dataset):
    def __init__(self, filepaths, seq_len=24, target_col='GPP_DT_VUT_REF', time_col='date',
                 forcing_cols=None, state_cols=None,
                 static_cols=['Lat', 'Long'], lc_col='Veg_ID',
                 feat_min_f=None, feat_max_f=None, feat_min_s=None, feat_max_s=None,
                 static_min=None, static_max=None, split_type='train'):

        self.seq_len = seq_len
        self.samples = []

        self.station_forcing = []
        self.station_state = []
        self.station_time_features = []
        self.station_targets = []
        self.station_dates = []

        self.station_static = []
        self.station_lc = []

        for filepath in filepaths:
            data = pd.read_csv(filepath)
            if time_col not in data.columns:
                print(f"⚠️ 在文件 {filepath} 中找不到时间列 '{time_col}'，跳过。")
                continue

            # 【修改后】异常值清洗：仅替换为 NaN 并直接剔除，取消线性插值
            data = data.replace([-9999, -9999.0, -999], np.nan)
            cols_to_clean = forcing_cols + state_cols + [target_col]
            cols_exist = [c for c in cols_to_clean if c in data.columns]

            # 必须保留 dropna：因为 PyTorch 无法计算 NaN，带 NaN 的行必须直接删掉
            data = data.dropna(subset=cols_exist).reset_index(drop=True)

            data[time_col] = pd.to_datetime(data[time_col])
            data = data.sort_values(by=time_col).reset_index(drop=True)

            if len(data) < seq_len:
                continue

            dates = data[time_col]
            self.station_dates.append(dates.values)

            hours = dates.dt.hour.values
            days = dates.dt.dayofyear.values
            time_feats = np.column_stack([
                np.sin(2 * np.pi * hours / 24.0), np.cos(2 * np.pi * hours / 24.0),
                np.sin(2 * np.pi * days / 365.25), np.cos(2 * np.pi * days / 365.25)
            ])

            forcing_data = data[forcing_cols].values
            state_data = data[state_cols].values
            static_data = data[static_cols].values
            lc_data = data[lc_col].values.astype(np.int64)

            if target_col in data.columns:
                targets = data[target_col].values
            else:
                targets = data.iloc[:, -1].values

            self.station_forcing.append(forcing_data)
            self.station_state.append(state_data)
            self.station_time_features.append(time_feats)
            self.station_targets.append(targets)
            self.station_static.append(static_data)
            self.station_lc.append(lc_data)

        if not self.station_forcing:
            raise ValueError(f"加载 {split_type} 数据失败，可能所有数据均被清洗掉或长度不足。")

        all_forcing_concat = np.vstack(self.station_forcing)
        all_state_concat = np.vstack(self.station_state)
        all_static_concat = np.vstack(self.station_static)

        self.feat_min_f = np.min(all_forcing_concat, axis=0) if feat_min_f is None else feat_min_f
        self.feat_max_f = np.max(all_forcing_concat, axis=0) if feat_max_f is None else feat_max_f
        self.feat_min_s = np.min(all_state_concat, axis=0) if feat_min_s is None else feat_min_s
        self.feat_max_s = np.max(all_state_concat, axis=0) if feat_max_s is None else feat_max_s

        self.static_min = np.min(all_static_concat, axis=0) if static_min is None else static_min
        self.static_max = np.max(all_static_concat, axis=0) if static_max is None else static_max

        # 归一化与样本切分 (包含时间连续性断层检测)
        for i in range(len(self.station_forcing)):
            self.station_forcing[i] = (self.station_forcing[i] - self.feat_min_f) / (self.feat_max_f - self.feat_min_f + 1e-8)
            self.station_state[i] = (self.station_state[i] - self.feat_min_s) / (self.feat_max_s - self.feat_min_s + 1e-8)
            self.station_static[i] = (self.station_static[i] - self.static_min) / (self.static_max - self.static_min + 1e-8)

            dates_array = self.station_dates[i]
            if len(dates_array) > 1:
                diffs = pd.Series(dates_array).diff()
                mode_step = diffs.mode()[0]
                expected_duration = mode_step * (self.seq_len - 1)

                num_samples = len(self.station_forcing[i]) - self.seq_len + 1
                if num_samples > 0:
                    for j in range(num_samples):
                        actual_duration = pd.Timedelta(dates_array[j + self.seq_len - 1] - dates_array[j])
                        if actual_duration == expected_duration:
                            self.samples.append((i, j))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        station_idx, start_idx = self.samples[idx]

        x_forcing = self.station_forcing[station_idx][start_idx : start_idx + self.seq_len]
        x_state = self.station_state[station_idx][start_idx : start_idx + self.seq_len]
        time_x = self.station_time_features[station_idx][start_idx : start_idx + self.seq_len]
        y = self.station_targets[station_idx][start_idx + self.seq_len - 1]
        target_date = self.station_dates[station_idx][start_idx + self.seq_len - 1]

        x_static = self.station_static[station_idx][start_idx : start_idx + self.seq_len]
        x_lc = self.station_lc[station_idx][start_idx : start_idx + self.seq_len]

        return (
            torch.tensor(x_forcing, dtype=torch.float32),
            torch.tensor(x_state, dtype=torch.float32),
            torch.tensor(time_x, dtype=torch.float32),
            torch.tensor(x_static, dtype=torch.float32),
            torch.tensor(x_lc, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            str(target_date)
        )

# ==========================================
# 2. TCN 基础模块定义
# ==========================================
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1, self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

# ==========================================
# 3. 交叉注意力融合模型
# ==========================================
class TCN_Transformer_CrossAttention(nn.Module):
    def __init__(self, num_forcing_features, num_state_features, seq_len,
                 num_static=2, num_lc_classes=13, lc_embed_dim=8,
                 d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super(TCN_Transformer_CrossAttention, self).__init__()

        self.tcn = TemporalConvNet(num_inputs=num_forcing_features,
                                   num_channels=[d_model] * 6,
                                   kernel_size=3, dropout=dropout)

        self.lc_embedding = nn.Embedding(num_embeddings=num_lc_classes, embedding_dim=lc_embed_dim)

        combined_state_dim = num_state_features + num_static + lc_embed_dim
        self.state_linear = nn.Linear(combined_state_dim, d_model)
        self.time_projector = nn.Linear(4, d_model)

        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.cross_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4), nn.ReLU(),
            nn.Linear(d_model // 4, 1)
        )

    def forward(self, x_forcing, x_state, time_x, x_static, x_lc):
        x_tcn_in = x_forcing.transpose(1, 2)
        f_tcn = self.tcn(x_tcn_in)
        f_met_memory = f_tcn.transpose(1, 2)

        lc_emb = self.lc_embedding(x_lc)
        combined_state = torch.cat([x_state, x_static, lc_emb], dim=-1)

        x_s_emb = self.state_linear(combined_state)
        time_emb = self.time_projector(time_x)
        x_state_combined = x_s_emb + time_emb
        f_state_global = self.transformer_encoder(x_state_combined)

        fused_features, _ = self.cross_attention(
            query=f_state_global,
            key=f_met_memory,
            value=f_met_memory
        )

        last_step_features = fused_features[:, -1, :]
        out = self.regressor(last_step_features)
        return out.squeeze(-1)

# ==========================================
# 4. SHAP 分析代码
# ==========================================
def perform_shap_analysis(model, dataloader, device, output_img_folder, forcing_cols, state_cols):
    print("\n🔍 开始执行 SHAP 变量重要性分析...")
    model.eval()

    shap_loader = DataLoader(dataloader.dataset, batch_size=4000, shuffle=True)
    batch = next(iter(shap_loader))

    batch_forcing, batch_state, batch_time = batch[0].to(device), batch[1].to(device), batch[2].to(device)
    batch_static, batch_lc = batch[3].to(device), batch[4].to(device)

    bg_size, test_size = 1000, 1000
    if batch_forcing.size(0) < (bg_size + test_size):
        print("⚠️ Batch size 太小，跳过 SHAP 分析。")
        return

    bg_f, bg_s = batch_forcing[:bg_size], batch_state[:bg_size]
    test_f, test_s = batch_forcing[bg_size:bg_size+test_size], batch_state[bg_size:bg_size+test_size]

    class SHAPWrapper(nn.Module):
        def __init__(self, base_model, t_ref, st_ref, lc_ref):
            super().__init__()
            self.base_model = base_model
            self.t_ref = t_ref[:1]
            self.st_ref = st_ref[:1]
            self.lc_ref = lc_ref[:1]

        def forward(self, x_f, x_s):
            b_size = x_f.size(0)
            t_x = self.t_ref.expand(b_size, -1, -1)
            x_st = self.st_ref.expand(b_size, -1, -1)
            x_l = self.lc_ref.expand(b_size, -1)
            out = self.base_model(x_f, x_s, t_x, x_st, x_l)
            return out.unsqueeze(-1)

    wrapper_model = SHAPWrapper(model, batch_time, batch_static, batch_lc).to(device)
    wrapper_model.eval()

    explainer = shap.GradientExplainer(wrapper_model, [bg_f, bg_s])
    shap_values = explainer.shap_values([test_f, test_s])

    if isinstance(shap_values, list) and len(shap_values) == 1 and isinstance(shap_values[0], list):
        shap_values = shap_values[0]

    shap_forcing = np.array(shap_values[0])
    shap_state = np.array(shap_values[1])

    shap_forcing_2d = shap_forcing.reshape(-1, len(forcing_cols))
    shap_state_2d = shap_state.reshape(-1, len(state_cols))

    test_f_2d = test_f.cpu().numpy().reshape(-1, len(forcing_cols))
    test_s_2d = test_s.cpu().numpy().reshape(-1, len(state_cols))

    shap_combined = np.concatenate([shap_forcing_2d, shap_state_2d], axis=1)
    features_combined = np.concatenate([test_f_2d, test_s_2d], axis=1)
    feature_names = forcing_cols + state_cols

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
    plt.title("SHAP Summary: Global Feature Impact on GPP (Time-Flattened)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Summary_Plot.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)
    plt.title("SHAP Global Feature Importance (Magnitude)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Bar_Plot.png"), dpi=300)
    plt.close()

    print(f"✅ SHAP 生成完成，保存至: {output_img_folder}")

# ==========================================
# 5. 主流程
# ==========================================
def                                                                                                                                                                                                                                                train_time_aware_model():
    seq_len = 96
    batch_size = 64
    num_epochs = 100
    learning_rate = 0.001
    patience = 10
    TIME_COLUMN_NAME = 'date'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data_folder = r"C:\Users\Admin\Desktop\实验四\全波段全变量DT"
    output_img_folder = os.path.join(data_folder, "全149站点各地类不超过8对照组")
    os.makedirs(output_img_folder, exist_ok=True)

    forcing_cols = ['SW_IN_F', 'SW_IN_POT', 'CO2_F_MDS', 'P_F', 'VPD_F', 'TA_F', 'TS_F_MDS_1', 'SWC_F_MDS_1', 'WS_F']
    state_cols = ['EPIC_Available_Mask', 'Band317nm_Ref', 'Band325nm_Ref', 'Band340nm_Ref',
                  'Band388nm_Ref', 'Band443nm_Ref', 'Band551nm_Ref', 'Band680nm_Ref',
                  'Band688nm_Ref', 'Band764nm_Ref', 'Band780nm_Ref', 'Mean_SZA', 'Mean_VZA', 'Mean_RAA']
    static_cols = ['Lat', 'Long']
    lc_col = 'Veg_ID'

    val_sites = [
        # --- 原始保留的验证集站点 ---
        'CR-Fsc_CRO', 'US-Mo1_CRO', 'FR-Aur_CRO', 'US-TKs_CSH', 'US-UC1_CVM',
        'CN-HeM_DBF', 'US-Rpf_DBF', 'KR-PcD_DBF', 'AU-Gin_EBF', 'US-xJE_ENF',
        'IL-Yat_ENF', 'DE-Har_ENF', 'FI-Ken_ENF', 'US-xKZ_GRA', 'IT-Tor_GRA',
        'US-Var_GRA', 'CH-Crs_GRA', 'KR-JjM_MF', 'US-xMB_OSH', 'ES-LM1_SAV',
        'JP-BBY_WET', 'AU-Cpr_WSA',
        # --- 为控制训练集同类站点不超过8个而新增的验证集站点 ---
        'DE-RuS_CRO', 'FR-CLt_GRA', 'FR-Nzn_GRA', 'IT-Lav_ENF', 'KR-AdC_ENF',
        'KR-Hnm_CRO', 'NL-Loo_ENF', 'NZ-Lau_GRA', 'RU-Fyo_ENF', 'US-Bi2_CRO',
        'US-Kon_GRA', 'US-NC2_ENF', 'US-Ne2_CRO', 'US-ONA_GRA', 'US-SP1_ENF',
        'US-Wkg_GRA', 'US-xBR_DBF', 'US-xDC_GRA', 'US-xSE_DBF', 'US-xTR_DBF'
    ]

    test_sites = [
        # --- 原始保留的测试集站点 ---
        'US-A39_CRO', 'CZ-KrP_CRO', 'AU-MvB_CRO', 'US-Rls_CSH', 'US-A37_CVM',
        'US-xUK_DBF', 'CZ-Lnz_DBF', 'FR-Fon_DBF', 'JP-Fhk_DNF', 'KR-WdE_EBF',
        'BE-Bra_ENF', 'US-xSB_ENF', 'US-MEF_ENF', 'US-xWR_ENF', 'AU-Sil_GRA',
        'BE-Dor_GRA', 'AU-Wel_GRA', 'US-KFS_GRA', 'CH-Lae_MF', 'DE-RuC_OSH',
        'US-xSJ_SAV', 'SE-Deg_WET', 'US-SRM_WSA',
        # --- 为控制训练集同类站点不超过8个而新增的测试集站点 ---
        'FI-Var_ENF', 'FR-Mej_GRA', 'FR-Tou_GRA', 'IT-SR2_ENF', 'KR-CRK_CRO',
        'KR-ScC_ENF', 'NL-Vkp_GRA', 'RU-Fy2_ENF', 'SE-Ros_ENF', 'US-CRK_ENF',
        'US-Me6_ENF', 'US-Ne1_CRO', 'US-Ne3_CRO', 'US-Ro1_CRO', 'US-SRG_GRA',
        'US-xAE_GRA', 'US-xCL_GRA', 'US-xDJ_ENF', 'US-xSL_CRO', 'US-xWD_GRA'
    ]
    all_files = glob.glob(os.path.join(data_folder, "*.[cC][sS][vV]"))
    if not all_files:
        print("❌ 错误：未在指定目录找到CSV文件。")
        return

    train_files, val_files, test_files = [], [], []
    for f in all_files:
        fname = os.path.basename(f)
        if any(site in fname for site in test_sites):
            test_files.append(f)
        elif any(site in fname for site in val_sites):
            val_files.append(f)
        else:
            train_files.append(f)

    print(f"📁 共找到 {len(all_files)} 个站点文件。")
    print(f"   -> 训练集: {len(train_files)} | 验证集: {len(val_files)} | 测试集: {len(test_files)}")

    train_dataset = TimeAwareMultiStationDataset(
        train_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col, split_type='train'
    )

    f_min_f, f_max_f = train_dataset.feat_min_f, train_dataset.feat_max_f
    f_min_s, f_max_s = train_dataset.feat_min_s, train_dataset.feat_max_s
    s_min, s_max = train_dataset.static_min, train_dataset.static_max

    val_dataset = TimeAwareMultiStationDataset(
        val_files, seq_len, time_col=TIME_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col,
        feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
        static_min=s_min, static_max=s_max, split_type='val'
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = TCN_Transformer_CrossAttention(
        num_forcing_features=len(forcing_cols),
        num_state_features=len(state_cols),
        seq_len=seq_len,
        num_static=len(static_cols),
        num_lc_classes=13,
        lc_embed_dim=8
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    checkpoint_latest_path = os.path.join(output_img_folder, "checkpoint_latest.pth")
    checkpoint_best_path = os.path.join(output_img_folder, "checkpoint_best.pth")

    start_epoch = 0
    best_val_loss = float('inf')
    epochs_no_improve = 0

    if os.path.exists(checkpoint_latest_path):
        print(f"\n🔄 检测到本地进度，尝试恢复...")
        try:
            checkpoint = torch.load(checkpoint_latest_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_loss = checkpoint['best_val_loss']
            epochs_no_improve = checkpoint['epochs_no_improve']
            print(f"✅ 从第 {start_epoch} 轮恢复，历史最佳 MSE: {best_val_loss:.4f}")
        except Exception as e:
            print(f"⚠️ 无法加载历史进度，将重新开始。错误信息: {e}")

    if start_epoch < num_epochs and epochs_no_improve < patience:
        print(f"🚀 开始训练...\n" + "-"*40)
        for epoch in range(start_epoch, num_epochs):
            model.train()
            train_loss = 0

            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in train_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                optimizer.zero_grad()
                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                loss = criterion(outputs, batch_y)
                loss.backward()

                # 梯度裁剪
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in val_loader:
                    batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                    batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                    outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            print(f"Epoch [{epoch+1:03d}/{num_epochs}] | Train MSE: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f}")

            if avg_val_loss < best_val_loss:
                print(f"   🌟 新的最佳模型！MSE: {avg_val_loss:.4f}。")
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), checkpoint_best_path)
            else:
                epochs_no_improve += 1
                print(f"   ⏳ 验证集未改善 (早停: {epochs_no_improve}/{patience})")

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'epochs_no_improve': epochs_no_improve
            }, checkpoint_latest_path)

            if epochs_no_improve >= patience:
                print(f"\n🛑 早停机制触发，训练结束。")
                break
            print("-" * 40)

# ==========================================
    # 6. 分站点独立测试评估 (整合高级绘图逻辑)
    # ==========================================
    if os.path.exists(checkpoint_best_path):
        print(f"\n🎯 开始测试集评估...")
        model.load_state_dict(torch.load(checkpoint_best_path, map_location=device))
    model.eval()

    global_all_preds, global_all_targets = [], []
    for test_file in test_files:
        station_name = os.path.basename(test_file).replace('.csv', '')
        single_test_dataset = TimeAwareMultiStationDataset(
            [test_file], seq_len, time_col=TIME_COLUMN_NAME,
            forcing_cols=forcing_cols, state_cols=state_cols,
            static_cols=static_cols, lc_col=lc_col,
            feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
            static_min=s_min, static_max=s_max, split_type='test'
        )

        if len(single_test_dataset) == 0: continue

        single_test_loader = DataLoader(single_test_dataset, batch_size=batch_size, shuffle=False)
        all_preds, all_targets, all_times = [], [], []

        with torch.no_grad():
            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, batch_dt in single_test_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc = batch_static.to(device), batch_lc.to(device)

                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(batch_y.numpy())
                all_times.extend(batch_dt)

        all_preds, all_targets = np.array(all_preds), np.array(all_targets)
        all_times = pd.to_datetime(all_times)

        # 应用数据清洗要求：剔除 GPP < 0 的异常值
        valid_mask = all_targets >= 0
        plot_targets = all_targets[valid_mask]
        plot_preds = all_preds[valid_mask]
        plot_times = all_times[valid_mask]

        if len(plot_targets) == 0:
            continue

        global_all_preds.extend(plot_preds)
        global_all_targets.extend(plot_targets)

        station_mse = np.mean((plot_preds - plot_targets)**2)
        station_r2 = r2_score(plot_targets, plot_preds)

        print(f"📢 站点: {station_name} | 测试集 MSE: {station_mse:.4f} | 测试集 R²: {station_r2:.4f}")

        # 图表 1: 滑动平均趋势图 (保持不变，仅修复黑底)
        window_size = 12
        all_targets_smooth = pd.Series(plot_targets).rolling(window=window_size, min_periods=1).mean().values
        all_preds_smooth = pd.Series(plot_preds).rolling(window=window_size, min_periods=1).mean().values

        plt.figure(figsize=(15, 6))
        plt.plot(plot_times, plot_targets, color='royalblue', linewidth=0.5, alpha=0.2, label='Actual (Raw)')
        plt.plot(plot_times, plot_preds, color='crimson', linewidth=0.5, alpha=0.2, label='Predicted (Raw)')
        plt.plot(plot_times, all_targets_smooth, label=f'Actual GPP (MA-{window_size})', color='royalblue', linewidth=1.5, alpha=0.9)
        plt.plot(plot_times, all_preds_smooth, label=f'Predicted GPP (MA-{window_size})', color='crimson', linewidth=1.5, linestyle='--', alpha=0.9)

        plt.title(f'[{station_name}] Test Set GPP Prediction (Moving Average Window={window_size})', fontsize=14, fontname='Arial')
        plt.xlabel('Time', fontsize=12, fontname='Arial')
        plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.grid(True, which='both', linestyle='--', alpha=0.5)
        plt.tight_layout()
        # 强制指定白色背景，关闭透明度
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_trend_ma.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # 图表 2: 局部放大图 (30天) —— 修改为使用 Raw Data
        zoom_days = 30
        steps_per_day = 24
        zoom_steps = zoom_days * steps_per_day
        zoom_steps = min(zoom_steps, len(plot_times))

        if zoom_steps > 0:
            peak_idx = np.argmax(all_targets_smooth)
            start_idx = max(0, peak_idx - zoom_steps // 2)
            end_idx = min(len(plot_times), start_idx + zoom_steps)

            if end_idx - start_idx < zoom_steps:
                start_idx = max(0, end_idx - zoom_steps)

            plt.figure(figsize=(15, 5))
            # 修改：此处换为 plot_targets 和 plot_preds（原始序列），以展示细节和高频变化
            plt.plot(plot_times[start_idx:end_idx], plot_targets[start_idx:end_idx],
                     label='Actual GPP (Raw)', color='royalblue', linewidth=1.5)
            plt.plot(plot_times[start_idx:end_idx], plot_preds[start_idx:end_idx],
                     label='Predicted GPP (Raw)', color='crimson', linewidth=1.5, linestyle='--', alpha=0.8)

            peak_date_str = pd.to_datetime(plot_times.iloc[peak_idx] if isinstance(plot_times, pd.Series) else plot_times[peak_idx]).strftime('%Y-%m-%d')

            plt.title(f'[{station_name}] 30-Day Zoomed Prediction (Raw Data Detail around {peak_date_str})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'})

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.6)
            plt.xticks(rotation=30)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_zoom_30days.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

        # 图表 3: 散点图
        plt.figure(figsize=(6, 6))
        plt.scatter(plot_targets, plot_preds, alpha=0.6, color='teal', s=15, edgecolors='k', linewidth=0.2)

        min_val = min(plot_targets.min(), plot_preds.min())
        max_val = max(plot_targets.max(), plot_preds.max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')

        plt.title(f'[{station_name}] Actual vs Predicted Scatter', fontname='Arial')
        plt.xlabel('Actual GPP', fontname='Arial')
        plt.ylabel('Predicted GPP', fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_scatter.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # ==========================================
        # 新增: 图表 4: 抽取任一年的数据展示时间序列
        # ==========================================
        years = plot_times.dt.year if hasattr(plot_times, 'dt') else pd.Series(plot_times).dt.year
        unique_years = years.unique()

        if len(unique_years) > 0:
            # 智能选择策略：为了图表丰满，默认选择数据点最多的一年，避免选到跨年只剩几天的数据
            target_year = years.value_counts().idxmax()
            year_mask = (years == target_year).values

            y_times = plot_times[year_mask]
            y_targets = plot_targets[year_mask]
            y_preds = plot_preds[year_mask]

            plt.figure(figsize=(15, 5))
            plt.plot(y_times, y_targets, label=f'Actual GPP ({target_year})', color='royalblue', linewidth=1.2, alpha=0.8)
            plt.plot(y_times, y_preds, label=f'Predicted GPP ({target_year})', color='crimson', linewidth=1.2, linestyle='--', alpha=0.8)

            plt.title(f'[{station_name}] Full Year GPP Time Series ({target_year})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'}, loc='upper right')

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.4)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_singleyear_{target_year}.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()
    # ------------------------------------------
    # 7. 全局指标汇总与 SHAP 分析
    # ------------------------------------------
    if len(global_all_targets) > 0:
        global_all_preds = np.array(global_all_preds)
        global_all_targets = np.array(global_all_targets)

        global_mse = np.mean((global_all_preds - global_all_targets)**2)
        global_r2 = r2_score(global_all_targets, global_all_preds)

        print("\n" + "="*50)
        print("🌎 所有站点测试集全局评估结果")
        print("="*50)
        print(f"有效总测试样本数 (剔除 GPP<0): {len(global_all_targets)}")
        print(f"Global Test MSE: {global_mse:.4f}")
        print(f"Global Test R² Score: {global_r2:.4f}")
        print("="*50)

        try:
            perform_shap_analysis(model, val_loader, device, output_img_folder, forcing_cols, state_cols)
        except Exception as e:
            pass

if __name__ == "__main__":
    train_time_aware_model()
#改为了局部归一化，目前最佳

📁 共找到 149 个站点文件。
   -> 训练集: 64 | 验证集: 42 | 测试集: 43
🚀 开始训练...
----------------------------------------
Epoch [001/100] | Train MSE: 7.3900 | Val MSE: 14.9485
   🌟 新的最佳模型！MSE: 14.9485。
----------------------------------------
Epoch [002/100] | Train MSE: 5.1779 | Val MSE: 18.5212
   ⏳ 验证集未改善 (早停: 1/10)
----------------------------------------
Epoch [003/100] | Train MSE: 4.6826 | Val MSE: 15.9717
   ⏳ 验证集未改善 (早停: 2/10)
----------------------------------------
Epoch [004/100] | Train MSE: 4.3830 | Val MSE: 15.8958
   ⏳ 验证集未改善 (早停: 3/10)
----------------------------------------
Epoch [005/100] | Train MSE: 4.1598 | Val MSE: 17.4791
   ⏳ 验证集未改善 (早停: 4/10)
----------------------------------------
Epoch [006/100] | Train MSE: 4.0019 | Val MSE: 17.4033
   ⏳ 验证集未改善 (早停: 5/10)
----------------------------------------
Epoch [007/100] | Train MSE: 3.8653 | Val MSE: 17.0326
   ⏳ 验证集未改善 (早停: 6/10)
----------------------------------------
Epoch [008/100] | Train MSE: 3.7397 | Val MSE: 15.8977
   ⏳ 验证

C:\Users\admin\AppData\Local\Temp\ipykernel_2016\3321063282.py:297: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
C:\Users\admin\AppData\Local\Temp\ipykernel_2016\3321063282.py:304: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)


✅ SHAP 生成完成，保存至: C:\Users\Admin\Desktop\实验四\全波段全变量DT\全149站点各地类不超过8对照组
